# APTOS MVP — bản sửa Review-2026-11-07

Sửa trực tiếp từ notebook 06/11; giữ 10 loss, A8 chính/A12 biến thể, S0 và checkpoint QWK trung bình 3 epoch.
Mặc định **CPU_TESTS**, không khởi chạy thực nghiệm thật. Trạng thái bàn giao **NOT READY cho MVP chính thức** cho đến khi có GPU preflight, đăng ký commit, BN comparison và các điều kiện ghi trong README.

Đã đọc [Review 07/11](https://docs.google.com/document/d/1kqn3fOBJ9D6O6ybT72SoE9T2r0gBTDhZ9igOX9Pp4_4/edit) và [Review 03/11](https://docs.google.com/document/d/1mrm9iikiA2RPcYlxBC1lBra2dZ0OY5iET0WuNol2j2g/edit). Không sửa bộ chia hoặc tự phê duyệt metadata S0.


## 2. CONFIG

CPU_TESTS → GPU_PREFLIGHT → SANITY (chọn từng tác vụ) → TUNE → MVP. RUN_ALL chỉ khi chủ động chọn.
`freeze_bn=None` nghĩa là chưa quyết định; kết quả `bn_selection.json` sẽ khóa True/False. `freeze_bn_affine=False` giữ gamma/beta được học.
A3 có hai chiều tuning; `boundary_A3_coordinate='gamma'` là đề xuất triển khai cần xác nhận bằng `boundary_policy_ref` trước TUNE.


In [1]:
# CONFIG QĐ25. CPU_TESTS chạy độc lập; các tác vụ GPU phải được chọn rõ ràng.
CFG = {
    'execution_mode': 'GPU_PREFLIGHT',  # CPU_TESTS | GPU_PREFLIGHT | SANITY | TUNE | MVP | RUN_ALL
    'input_root': '/kaggle/input', 'output_root': '/kaggle/working/APTOS_MVP_S0v1',
    'allow_exploratory_unapproved': False, 'allow_provisional_protocol': False,
    'auto_resume': True, 'require_dual_t4': True,
    'approved_split_sha256': 'd53328d2a012c6b32d658b89494092b5deaf27b90258191362dc1695394e27ca',
    'split_sha256_selector': None,  # chỉ cần khi có nhiều bộ S0 khác nhau
    'seed': 42,  # QĐ25
    'backbone': 'resnet50', 'pretrained': True, 'input_size': 384,
    'batch_size': 32, 'accumulation_steps': 1, 'num_workers': 2,
    'epochs': 50, 'patience': 10, 'lr': 1e-4, 'weight_decay': 1e-4,
    'optimizer': 'AdamW', 'scheduler': 'ReduceLROnPlateau',
    'scheduler_factor': 0.5, 'scheduler_patience': 3,
    'amp': True, 'grad_clip_norm': 5.0, 'freeze_bn': None,  # khóa từ BN comparison
    'freeze_bn_affine': False,  # False: gamma/beta vẫn có gradient
    'sanity_overfit_loss_tolerance': 0.01,  # ngưỡng kỹ thuật cho near-zero; ghi rõ trong log
    'sanity_tasks': [],  # chọn: overfit, shuffled, bn, runtime, probe
    'a0_ce_backend': 'native',  # chỉ đổi sang logsoftmax_gather nếu CUDA preflight báo lỗi CE/NLL
    'boundary_enabled': True, 'max_boundary_trials': 6,
    'boundary_A3_coordinate': 'gamma',  # 1 trial/method; mở gamma, giữ beta của trial tốt nhất
    'boundary_policy_ref': None,  # cần xác nhận quy tắc A3 hai chiều trước TUNE
    'registration_path': None,  # runtime_registration.json tạo SAU commit, không commit file này
    'checkpoint_smoothing': 3, 'crop_threshold': 8,
    'brightness': 0.1, 'contrast': 0.1,
    'hflip_p': 0.5, 'vflip_p': 0.5, 'rotation_degrees': 15.0,
    'normalization_mean': [0.485, 0.456, 0.406],
    'normalization_std': [0.229, 0.224, 0.225],
    'sqrt_beta': 0.999, 'sqrt_gamma': 2.0,
    'lambda_grid_A8': [0.1, 0.3, 0.5, 1.0, 3.0, 10.0],
    'lambda_grid_A6_A9': [0.1, 0.3, 0.5, 1.0, 3.0, 10.0],
    'alpha_grid_A12': [0.25, 0.5, 1.0, 2.0, 4.0],
    'beta_grid_A3': [0.99, 0.999, 0.9999], 'gamma_grid_A3': [1.0, 2.0],
    'slace_alpha_grid': [0.3, 0.8, 1.0, 3.0, 10.0, 25.0],
    'slace_variant': 'prox_max',
    'bootstrap_reps': 2000, 'bootstrap_seed': 20261105,
    'session_hours': 10.0, 'stop_margin_minutes': 20,
    'cache_images': True, 'telemetry_interval_seconds': 10,
    'metric_max_mb': 32, 'git_commit': None, 'kaggle_version': None,
    'protocol_approved_ref': 'Review-2026-11-07 – QĐ24, QĐ25',
}
METHODS = ('A0','A3','A4','A6','A8','A9','A11','A12','SLACE','CORN')
K = 5
IMPLEMENTATION_CHOICES = [
 'Backbone ResNet50 ImageNet, resize 384, crop RGB>8, augmentation và chuẩn hóa ImageNet',
 'AdamW/lr/weight decay/epochs/patience/batch/BN và scheduler trong CONFIG',
 'gamma=2 cho sqrt-CB-Focal; lấy căn raw effective-number rồi chuẩn hóa mean class=1',
 'A6/A9 dùng cùng grid lambda A8; SLACE chọn 6 alpha từ grid gốc',
 'Squared EMD lấy trung bình K-1 ngưỡng; seed train 42 (QĐ25)',
 'Checkpoint QWK trung bình trượt 3 epoch, hòa chọn NLL thấp hơn; T dùng log-probability',
 'Bootstrap ảnh theo QĐ19; thêm cluster bootstrap để kiểm tra phụ thuộc trùng ảnh',
 'CORN dùng tổng xác suất vượt ngưỡng >0.5 cho bảng chính; argmax chỉ phụ',
 'A3 có lưới 2 chiều: đề xuất mở gamma ×/÷3, giữ beta; cần boundary_policy_ref trước tuning',
]


## 3. Môi trường, log và metric.json

Yêu cầu PyTorch có `torch.amp.GradScaler`, torchvision, NumPy, pandas, Pillow, scipy, scikit-learn, psutil, tqdm và matplotlib. Không tự cài package bằng Internet. Trọng số ImageNet-1K V2 có thể được tự tải qua Internet ở lần chạy đầu tiên. `metric.json` được tạo ở đầu `main()`, cập nhật atomically và đọc lại sau mỗi mốc/epoch. Nếu import package bắt buộc thất bại thì kernel báo lỗi ngay trước khi vào main.

JSON chỉ chứa tổng hợp. Danh sách ID, dự đoán, đường dẫn ảnh và hash pixel từng ảnh nằm ngoài JSON. `training.log` giữ traceback và đường dẫn kỹ thuật khi cần. RAM là RSS tiến trình, GPU được đo từng thiết bị; giá trị không đọc được là null/unavailable. Không gộp VRAM hai GPU thành một GPU.

In [2]:
NOTEBOOK_CODE_SHA256 = 'ddc6bbeef79e03487ae7d41626d555b8871132f1449f04a9f799986b9220ca8a'
import os, sys, json, math, time, random, hashlib, logging, traceback, platform, ast, types, inspect
import shutil, subprocess, tempfile, threading, copy, gc, warnings
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageEnhance
import torch
torch.set_num_threads(min(4,os.cpu_count() or 1))
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision
from torchvision import models, transforms
import psutil
from tqdm.auto import tqdm
from sklearn.metrics import confusion_matrix, cohen_kappa_score, precision_recall_fscore_support
from scipy.optimize import minimize_scalar
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

class Blocked(RuntimeError): pass
class SessionLimit(RuntimeError): pass
def require(ok, msg):
    if not ok: raise Blocked(msg)
def clean(obj):
    if isinstance(obj, dict): return {str(k):clean(v) for k,v in obj.items()}
    if isinstance(obj, (list,tuple,np.ndarray)): return [clean(v) for v in obj]
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.bool_,)): return bool(obj)
    if isinstance(obj, (float,np.floating)): return float(obj) if np.isfinite(obj) else None
    if isinstance(obj, Path): return str(obj)
    return obj
def digest(obj):
    return hashlib.sha256(json.dumps(clean(obj),sort_keys=True,ensure_ascii=False).encode()).hexdigest()
def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''): h.update(b)
    return h.hexdigest()
def read_json(p): return json.loads(Path(p).read_text(encoding='utf-8-sig'))
def atomic_json(p, value):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_name(p.name+'.tmp')
    with open(tmp,'w',encoding='utf8') as f:
        json.dump(clean(value),f,ensure_ascii=False,allow_nan=False,separators=(',',':'))
        f.flush();os.fsync(f.fileno())
    os.replace(tmp,p)
def atomic_torch(p, value):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);tmp=p.with_name(p.name+'.tmp')
    torch.save(value,tmp);os.replace(tmp,p)
def atomic_npz(destination, **arrays):
    p=Path(destination);p.parent.mkdir(parents=True,exist_ok=True);tmp=p.with_name(p.name+'.tmp')
    with open(tmp,'wb') as f: np.savez_compressed(f,**arrays)
    os.replace(tmp,p)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True)
def rng_state():
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
            'cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}
def restore_rng(s):
    random.setstate(s['python']);np.random.set_state(s['numpy']);torch.set_rng_state(s['torch'].cpu())
    if s['cuda']: torch.cuda.set_rng_state_all([v.cpu() for v in s['cuda']])

FORBIDDEN_KEYS={'id_code','image_id','image_ids','images','paths','path','filenames','filename',
 'pixel_sha256','y_true','y_pred','probabilities','predictions','sample_ids','checkpoint_path','artifacts'}
KNOWN_IDS=set()
def metric_privacy_check(obj):
    if isinstance(obj,dict):
        for k,v in obj.items():
            require(str(k).lower() not in FORBIDDEN_KEYS,'Khóa per-sample bị cấm trong metric.json')
            require(str(k) not in KNOWN_IDS,'ID ảnh trong khóa metric.json')
            metric_privacy_check(v)
    elif isinstance(obj,list):
        for v in obj: metric_privacy_check(v)
    elif isinstance(obj,str):
        import re
        require(not re.search(r'\.(png|jpg|jpeg|tiff|bmp)(?:\b|$)',obj,re.I),'Tên ảnh trong metric.json')
        require(not obj.startswith(('/kaggle/','/workspace/')),'Đường dẫn trong metric.json')
        require(not (set(re.findall(r'\b[0-9a-f]{12}\b',obj)) & KNOWN_IDS),'ID ảnh trong metric.json')
def write_metric():
    metric_privacy_check(METRIC)
    out=clean(METRIC)
    require(len(json.dumps(out).encode())<CFG['metric_max_mb']*1024**2,'metric.json vượt giới hạn')
    require(set(('schema_version','experiment','hardware','methods','execution_summary'))<=set(out),'Schema sai')
    atomic_json(OUT/'metric.json',out)
    require(read_json(OUT/'metric.json')['schema_version']=='1.0','Không đọc lại được metric.json')
def initialize_output():
    global OUT, LOG, METRIC, START_TIME
    START_TIME=time.monotonic();OUT=Path(CFG['output_root']);OUT.mkdir(parents=True,exist_ok=True)
    LOG=logging.getLogger('aptos_mvp');LOG.handlers.clear();LOG.setLevel(logging.INFO)
    fmt=logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
    for h in [logging.StreamHandler(sys.stdout),logging.FileHandler(OUT/'training.log',encoding='utf8')]:
        h.setFormatter(fmt);LOG.addHandler(h)
    old=OUT/'metric.json'
    METRIC=read_json(old) if old.exists() else {
        'schema_version':'1.0','experiment':{'status':'not_started'},'hardware':{},
        'methods':{m:{'status':'not_run','tuning':{},'folds':{}} for m in METHODS},
        'oof_summary':{},'comparisons':{},'hypothesis_tests':{},'tests':{},
        'execution_summary':{'completed_runs':0,'failed_runs':0,'total_runtime_seconds':0}}
    write_metric()
def hardware_info():
    g=[]
    for i in range(torch.cuda.device_count()):
        p=torch.cuda.get_device_properties(i)
        g.append({'index':i,'model':p.name,'vram_gb':p.total_memory/1024**3})
    cpu=platform.processor()
    if Path('/proc/cpuinfo').exists():
        cpu=next((l.split(':',1)[1].strip() for l in Path('/proc/cpuinfo').read_text().splitlines() if l.startswith('model name')),cpu)
    driver=None
    try: driver=subprocess.check_output(['nvidia-smi','--query-gpu=driver_version','--format=csv,noheader'],text=True,timeout=5).strip().splitlines()[0]
    except (OSError,subprocess.SubprocessError): pass
    return {'cpu':{'model':cpu,'logical_cores':os.cpu_count(),'ram_total_gb':psutil.virtual_memory().total/1024**3},
            'gpus':g,'gpu_count':len(g),'cuda':torch.version.cuda,'driver':driver,
            'torch':torch.__version__,'torchvision':torchvision.__version__,'python':platform.python_version(),
            'packages':{p:version(p) for p in ['numpy','pandas','Pillow','scipy','scikit-learn','psutil','tqdm','matplotlib']},
            'multi_gpu_backend':'DataParallel' if len(g)>=2 else None}

class Telemetry:
    def __init__(self): self.rows=[];self.stop=threading.Event()
    def sample(self):
        try:cpu=psutil.cpu_percent()
        except (psutil.Error,OSError):cpu=None
        try:ram=psutil.Process().memory_info().rss/1024**3
        except (psutil.Error,OSError):ram=None
        row={'cpu_utilization':cpu,'ram_gb':ram,'gpus':[]}
        try:
            s=subprocess.check_output(['nvidia-smi','--query-gpu=index,utilization.gpu,memory.used,temperature.gpu,power.draw','--format=csv,noheader,nounits'],text=True,timeout=3)
            for line in s.strip().splitlines():
                vals=line.split(',');r={}
                for k,v in zip(['index','utilization','memory_mb','temperature_c','power_w'],vals):
                    try:r[k]=float(v)
                    except ValueError:r[k]=None
                row['gpus'].append(r)
        except (OSError,subprocess.SubprocessError): pass
        self.rows.append(row)
    def loop(self):
        while not self.stop.wait(CFG['telemetry_interval_seconds']):self.sample()
    def __enter__(self):
        self.sample();self.thread=threading.Thread(target=self.loop,daemon=True);self.thread.start();return self
    def __exit__(self,*args):self.stop.set();self.thread.join(timeout=4);self.sample()
    def summary(self):
        def stats(vals):
            v=[x for x in vals if x is not None]
            return {'mean':float(np.mean(v)) if v else None,'max':float(max(v)) if v else None,'status':'measured' if v else 'unavailable'}
        g=[]
        for i in range(torch.cuda.device_count()):
            rows=[r for x in self.rows for r in x['gpus'] if r.get('index')==i]
            d={'index':i,**{k:stats([r.get(k) for r in rows]) for k in ['utilization','memory_mb','temperature_c','power_w']}}
            d.update({k:getattr(torch.cuda,k)(i)/1024**3 for k in ['memory_allocated','memory_reserved','max_memory_allocated','max_memory_reserved']});g.append(d)
        return {'cpu_utilization':stats([r['cpu_utilization'] for r in self.rows]),'ram_gb':stats([r['ram_gb'] for r in self.rows]),'gpus':g}

def update_resource_totals(runs):
    rows=[h for r in runs for h in r.get('history',[])]
    def peak(values):
        v=[x for x in values if x is not None];return max(v) if v else None
    METRIC['hardware'].setdefault('cpu',{})['ram_peak_gb']=peak([h['resources']['ram_gb']['max'] for h in rows])
    train_sec=sum(h['train_seconds'] for h in rows);count=sum(h['train']['samples'] for h in rows)
    METRIC['execution_summary'].update({'training_seconds':train_sec,
        'validation_seconds':sum(h['validation_seconds'] for h in rows),
        'training_throughput_images_per_second':count/train_sec if train_sec else None})
    for gpu in METRIC['hardware'].get('gpus',[]):
        readings=[g for h in rows for g in h['resources']['gpus'] if g['index']==gpu['index']]
        gpu['peak_allocated_gb']=peak([g['max_memory_allocated'] for g in readings])
        gpu['peak_reserved_gb']=peak([g['max_memory_reserved'] for g in readings])
        gpu['utilization_max']=peak([g['utilization']['max'] for g in readings])
        means=[g['utilization']['mean'] for g in readings if g['utilization']['mean'] is not None]
        gpu['utilization_mean_of_epoch_means']=float(np.mean(means)) if means else None

RUN_SEED=CFG['seed']
PROTOCOL_HASH=None
SELECTION_LOCK=None


## 4. Tự tìm S0 và kiểm toán partition

Tìm theo nội dung `splits.json` và manifest đi kèm, đối chiếu schema thực của S0. Nếu nhiều bộ khác hash, dừng. Xác thực ID, nhãn, exclusions, ba loại cluster, pixel SHA đã biết, số ảnh/grade/cluster từng partition, coverage test đúng một lần, hash dữ liệu và masks. Khi RUN_ALL, đọc `train.csv` đúng hash và kiểm tra bytes toàn bộ ảnh so với manifest.

Không suy ra phân tách theo bệnh nhân: S0 không có định danh bệnh nhân. Kiểm tra này chỉ bảo đảm theo các nhóm đã xác nhận trong bộ kiểm toán.

In [3]:
def discover_s0(root):
    candidates = []
    for dirpath, dirnames, filenames in os.walk(root):
        # Không quét vào archive của bất kỳ dataset nào
        dirnames[:] = sorted(
            d for d in dirnames
            if d.lower() != 'archive'
        )
        if 'splits.json' not in filenames:
            continue
        p = Path(dirpath) / 'splits.json'
        try:
            s=read_json(p);r=p.parent.parent
            if not (isinstance(s.get('scenarios'),dict) and s.get('main_scenario') in s['scenarios']):continue
            m=read_json(r/'reproducibility/s0_version_manifest.json')
            require(sha_file(p)==m['splits_json_sha256'],'Hash splits.json không khớp: '+str(p))
            if CFG['split_sha256_selector'] and m['splits_json_sha256']!=CFG['split_sha256_selector']:continue
            candidates.append((r,s,m))
        except (KeyError,ValueError,FileNotFoundError) as e:
            LOG.warning('Bỏ ứng viên thiếu schema: %s (%s)',p,type(e).__name__)
    # Hai mount cùng hash được chấp nhận chỉ khi toàn bộ tệp đầu vào cần thiết giống nhau.
    unique={}
    for r,s,m in candidates:
        sig=(m['splits_json_sha256'],sha_file(r/'reproducibility/file_hashes.json'))
        if sig in unique:
            previous=unique[sig][0]
            for n in ['splits','reproducibility','reports','data_quality']:
                for f in (r/n).glob('*'):
                    if f.is_file() and (previous/n/f.name).is_file():
                        require(sha_file(f)==sha_file(previous/n/f.name),'Hai bản S0 cùng hash nhưng metadata khác')
        else:unique[sig]=(r,s,m)
    require(len(unique)==1,f'Có {len(unique)} bộ S0 hợp lệ khác nhau; attach duy nhất một bộ hoặc chỉ định hash trong CONFIG.')
    return next(iter(unique.values()))

def audit_s0(root, split, manifest, source_table=None):
    scenario=split['main_scenario'];folds=split['scenarios'][scenario]
    config=read_json(root/'reproducibility/config.json')
    provenance=read_json(root/'reproducibility/provenance.json')
    fingerprints=read_json(root/'reproducibility/input_fingerprints.json')
    hashes=read_json(root/'reproducibility/file_hashes.json')['sha256']
    required=['splits/splits.json','splits/clusters.csv','splits/partition_counts.csv',
        'splits/partition_membership.csv','splits/split_report.json','splits/acquisition_groups.csv',
        'splits/sensitivity_test_masks_preregistered.csv','data_quality/image_manifest.csv',
        'data_quality/excluded_images.csv','reproducibility/config.json','reproducibility/provenance.json',
        'reproducibility/input_fingerprints.json','reproducibility/s0_version_manifest.json',
        'reproducibility/sensitivity_preregistration.json','reports/validation_checks.json',
        'reports/unit_tests.json','reports/dataset_audit.json']
    for n in required:
        require(n in hashes and (root/n).is_file(),'Thiếu tệp hoặc hash: '+n)
        require(sha_file(root/n)==hashes[n],'Checksum sai: '+n)
    for n,expected in hashes.items():
        if n.startswith(('splits/','reproducibility/')) and (root/n).is_file():
            require(sha_file(root/n)==expected,'Metadata checksum sai: '+n)
    require(sha_file(root/'splits/splits.json')==manifest['splits_json_sha256'],'Split hash sai')
    require(digest(split['scenarios'])==provenance['split_assignments_sha256'],'Assignment hash sai')
    require(digest(config)==provenance['config_sha256'],'Config S0 hash sai')
    require(split['version']==manifest['version'],'Version S0 mâu thuẫn')
    # QĐ24 changes manifest status while splits.json bytes remain frozen.
    require(split['status']==manifest['status'] or (manifest['status']=='approved_frozen' and bool(manifest.get('advisor_approval_ref'))),'Status S0 không có phê duyệt chuyển trạng thái')
    require(provenance==split['provenance'],'Provenance mâu thuẫn')
    require(len(folds)==config['n_folds'],'Số fold mâu thuẫn')
    require(len({f['fold'] for f in folds})==len(folds),'Lặp chỉ số fold')
    checks=read_json(root/'reports/validation_checks.json')['checks']
    require(all(c['passed'] is True for c in checks),'Kiểm toán S0 còn lỗi')
    require(all(c['passed'] is True for c in read_json(root/'reports/unit_tests.json')),'S0 unit tests thất bại')
    audit=read_json(root/'reports/dataset_audit.json')
    require(audit['all_automated_checks_passed'] is True,'S0 automated audit chưa đạt')
    require(audit['version']==manifest['version'],'Audit version mâu thuẫn')
    # Historical audit stays immutable; current approval/archive status comes from manifest.
    require(sha_file(root/'reproducibility/sensitivity_preregistration.json')==manifest['sensitivity_preregistration_sha256'],'Manifest sensitivity hash sai')
    im=pd.read_csv(root/'data_quality/image_manifest.csv',dtype={'id_code':str}).set_index('id_code')
    cl=pd.read_csv(root/'splits/clusters.csv',dtype={'id_code':str}).set_index('id_code')
    ac=pd.read_csv(root/'splits/acquisition_groups.csv',dtype={'id_code':str}).set_index('id_code')
    ex=pd.read_csv(root/'data_quality/excluded_images.csv',dtype={'id_code':str})
    counts=pd.read_csv(root/'splits/partition_counts.csv')
    membership=pd.read_csv(root/'splits/partition_membership.csv',dtype={'id_code':str})
    reports=read_json(root/'splits/split_report.json')[scenario]
    masks=pd.read_csv(root/'splits/sensitivity_test_masks_preregistered.csv',dtype={'id_code':str})
    prereg=read_json(root/'reproducibility/sensitivity_preregistration.json')
    require(prereg['locked_before_MVP'] and prereg['no_resplitting'],'Sensitivity chưa khóa')
    require(digest(masks.to_dict('records'))==prereg['masks_sha256'],'Sensitivity logical hash sai')
    require(len(masks)==prereg['n_masks'],'Mask count sai')
    require(not im.index.duplicated().any() and not cl.index.duplicated().any() and not ac.index.duplicated().any(),'Lặp ID metadata')
    require(set(im.index)==set(cl.index)==set(ac.index),'Inventory metadata khác nhau')
    require(np.array_equal(im.grade,cl.loc[im.index].grade) and np.array_equal(im.grade,ac.loc[im.index].grade),'Nhãn metadata mâu thuẫn')
    if source_table is not None:
        require(set(source_table.index)==set(im.index),'train.csv khác inventory S0')
        require(np.array_equal(source_table.loc[im.index].diagnosis,im.grade),'Nhãn train.csv khác S0')
    fp=digest({'csv':read_json(root/'reports/input_inventory.json')['csv_sha256'],
        'images':im.reset_index()[['id_code','source_file_sha256']].values.tolist(),
        'preprocessing':fingerprints['preprocessing_fingerprint']})
    require(fp==manifest['dataset_fingerprint']==fingerprints['dataset_fingerprint']==provenance['dataset_fingerprint'],'Dataset fingerprint mâu thuẫn')
    excluded=set(ex.id_code);eligible=set(cl.index[~cl.excluded])
    require(set(cl.index[cl.excluded])==excluded,'Exclusion metadata mâu thuẫn')
    require(len(eligible)==manifest['eligible_images']==audit['eligible_n']==reports['n_images'],'Eligible count sai')
    require({str(k):int(v) for k,v in im.loc[sorted(eligible)].grade.value_counts().sort_index().items()}==manifest['eligible_grade_counts'],'Grade counts sai')
    require(im.loc[sorted(eligible)].pixel_sha256.nunique()==reports['n_unique_pixels'],'Unique pixel count sai')
    require(cl.loc[sorted(eligible)].s0_cluster_id.nunique()==reports['n_clusters'],'Cluster count sai')
    test_all=[];summary=[]
    for f in folds:
        groups={p:set(f[p]) for p in ['train','validation','test']}
        require(set.union(*groups.values())==eligible,'Fold không phủ eligible inventory')
        for p,ids in groups.items():
            require(ids and len(ids)==len(f[p]) and ids<=set(im.index),'ID lặp/không tồn tại')
            require(not ids&excluded,'ID đã loại xuất hiện trong split')
            for q,other in groups.items():
                if p>=q:continue
                require(not ids&other,'Rò rỉ ID')
                for col in ['exact_cluster_id','identity_cluster_id','s0_cluster_id']:
                    require(not set(cl.loc[sorted(ids),col])&set(cl.loc[sorted(other),col]),'Rò rỉ cluster '+col)
                require(not set(im.loc[sorted(ids)].pixel_sha256)&set(im.loc[sorted(other)].pixel_sha256),'Rò rỉ pixel SHA256')
            cr=counts[(counts.scenario==scenario)&(counts.fold==f['fold'])&(counts.partition==p)]
            require(len(cr)==1,'Partition count không duy nhất');cr=cr.iloc[0]
            nr=next(x for x in reports['folds'] if x['fold']==f['fold'])['partitions'][p]
            gcnt=np.bincount(im.loc[sorted(ids)].grade,minlength=K)
            require(len(ids)==int(cr.n_images)==nr['n_images'],'Partition size sai')
            require(gcnt.tolist()==[int(cr[f'grade_{k}']) for k in range(K)],'Partition grade sai')
            require({str(k):int(gcnt[k]) for k in range(K)}==nr['grade']['counts'],'Split report grade sai')
            require(cl.loc[sorted(ids)].s0_cluster_id.nunique()==int(cr.n_clusters)==nr['n_clusters'],'Partition cluster sai')
            require(im.loc[sorted(ids)].pixel_sha256.nunique()==nr['n_unique_pixels'],'Partition unique pixel sai')
            # S0-v1 membership schema chỉ có fold/partition/id_code/cluster_id, không có scenario.
            mm=membership[(membership.fold==f['fold'])&(membership.partition==p)]
            require(len(mm)==len(ids) and set(mm.id_code)==ids,'Partition membership sai')
            require(np.array_equal(mm.set_index('id_code').loc[sorted(ids)].cluster_id,
                cl.loc[sorted(ids)].s0_cluster_id),'Membership cluster sai')
            summary.append({'fold':f['fold'],'partition':p,'n':len(ids),'grade_counts':gcnt.tolist(),'clusters':int(cr.n_clusters)})
        fm=masks[masks.fold==f['fold']]
        require(len(fm)==len(groups['test']) and set(fm.id_code)==groups['test'],'Sensitivity mask không khớp test')
        expected=set(im.loc[sorted(groups['test'])].reset_index().drop_duplicates('pixel_sha256').id_code)
        require(set(fm.loc[fm.unique_pixel_test_keep,'id_code'])==expected,'Unique-pixel mask sai')
        require((fm.phash4_filtered_test_keep==~fm.unverified_phash_le4_to_train).all(),'Mask pHash sai')
        test_all.extend(f['test'])
    require(len(test_all)==len(eligible) and set(test_all)==eligible,'Test folds phải phủ đúng một lần')
    approved=(manifest.get('official') is True and manifest['status']=='approved_frozen' and bool(manifest.get('advisor_approval_ref')))
    if not approved and CFG['execution_mode'] in ('TUNE','MVP','RUN_ALL'):
        require(False,'S0 manifest chưa approved_frozen/official; không sửa splits.json')
    notices=[]
    if sha_file(root/'README.md')!=hashes.get('README.md'):
        notices.append('README changed after hashed export; canonical data hashes passed')
        LOG.warning('README khác hash lưu trước đó. Dùng trạng thái từ manifest; không tự sửa metadata.')
    if not manifest.get('legacy_archived'):notices.append('legacy_archived=false')
    return {'root':root,'split':split,'manifest':manifest,'config':config,'folds':folds,'scenario':scenario,
        'images':im,'clusters':cl,'acquisition':ac,'masks':masks,'eligible':eligible,
        'summary':{'version':manifest['version'],'status':manifest['status'],'official':manifest.get('official',False),
        'approval_present':bool(manifest.get('advisor_approval_ref')),'legacy_archived':manifest.get('legacy_archived'),
        'result_scope':'approved' if approved else 'exploratory_unapproved','scenario':scenario,
        'fold_count':len(folds),'eligible_count':len(eligible),'excluded_count':len(excluded),
        'unique_pixel_count':reports['n_unique_pixels'],'cluster_count':reports['n_clusters'],
        'split_sha256':manifest['splits_json_sha256'],'dataset_fingerprint':fp,'partitions':summary,
        'notices':notices,'patient_level_split':split['patient_level_split']}}

def find_source(root,s0):
    expected=read_json(s0['root']/'reports/input_inventory.json')['csv_sha256']
    candidates=[]
    for p in Path(root).rglob('train.csv'):
        table=pd.read_csv(p,dtype={'id_code':str})
        if set(['id_code','diagnosis'])<=set(table) and sha_file(p)==expected:candidates.append((p,table))
    require(candidates,'Không tìm được train.csv đúng SHA-256 S0')
    require(len(candidates)==1,'Nhiều bản train.csv; bỏ mount trùng để chọn nguồn duy nhất')
    p,table=candidates[0];require(not table.id_code.duplicated().any(),'train.csv lặp ID')
    index=defaultdict(list)
    for f in Path(root).rglob('*'):
        if f.is_file() and f.suffix.lower() in {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'} and f.stem in set(table.id_code):index[f.stem].append(f)
    require(all(len(index[i])==1 for i in table.id_code),'Thiếu ảnh hoặc nhiều file cho cùng ID; chỉ attach một nguồn ảnh APTOS')
    paths={i:index[i][0] for i in table.id_code}
    # Kiểm tra bytes gốc, không chỉ tin tên ảnh. Có thể mất vài phút trên toàn bộ source.
    for i in tqdm(table.id_code,desc='SHA256 ảnh nguồn',mininterval=3):
        require(sha_file(paths[i])==s0['images'].loc[i,'source_file_sha256'],'Ảnh nguồn khác nội dung đã kiểm toán')
    return table.set_index('id_code'),paths



## 5. Dữ liệu và phê duyệt

Đọc nguyên S0-v1.0; checksum và cluster leakage audit vẫn được giữ. Không chia lại.
QĐ24 cho phép cập nhật trạng thái manifest sau khi splits.json đã khóa: do đó trạng thái lịch sử trong splits/audit có thể cũ; trạng thái hiện tại lấy từ manifest. GPU preflight có thể kiểm tra bản S0 chưa cập nhật trạng thái, nhưng TUNE/MVP yêu cầu manifest chính thức.


## 6. Dataset → crop → pad → resize → augmentation → normalize

Giữ nguyên crop RGB > 8, pad và resize. Sau flip/rotation, thêm brightness và contrast ±10%, cùng RNG từ (seed, epoch, image_id). Validation/test không có random augmentation.


In [4]:
class FundusDataset(Dataset):
    def __init__(self,ids,s0,paths,training=False,epoch=0):
        self.ids=list(ids);self.s0=s0;self.paths=paths;self.training=training;self.epoch=epoch
        self.normalize=transforms.Normalize(CFG['normalization_mean'],CFG['normalization_std'])
        self.cache=OUT/'cache'/digest([s0['summary']['dataset_fingerprint'],CFG['input_size'],CFG['crop_threshold']])[:16]
        self.cache.mkdir(parents=True,exist_ok=True)
    def __len__(self):return len(self.ids)
    def __getitem__(self,j):
        iid=self.ids[j];cp=self.cache/(iid+'.npy')
        if CFG['cache_images'] and cp.exists(): a=np.load(cp,allow_pickle=False)
        else:
            with Image.open(self.paths[iid]) as img:a=np.asarray(img.convert('RGB'))
            mask=a.max(2)>CFG['crop_threshold'];ys,xs=np.where(mask)
            require(len(xs)>0,'Ảnh không có vùng fundus sau crop')
            img=Image.fromarray(a[ys.min():ys.max()+1,xs.min():xs.max()+1])
            side=max(img.size);img=ImageOps.pad(img,(side,side),color=(0,0,0))
            a=np.asarray(img.resize((CFG['input_size'],CFG['input_size']),Image.Resampling.BILINEAR)).copy()
            if CFG['cache_images']:
                tmp=cp.with_name(cp.name+f'.{os.getpid()}.tmp')
                with open(tmp,'wb') as h:np.save(h,a,allow_pickle=False)
                os.replace(tmp,cp)
        img=Image.fromarray(a)
        if self.training:
            # Cùng ảnh/fold/epoch -> cùng augmentation, không phụ thuộc thứ tự phương pháp hoặc worker.
            seed=int(hashlib.sha256(f'{RUN_SEED}:{self.epoch}:{iid}'.encode()).hexdigest()[:16],16)
            rnd=random.Random(seed)
            if rnd.random()<CFG['hflip_p']:img=ImageOps.mirror(img)
            if rnd.random()<CFG['vflip_p']:img=ImageOps.flip(img)
            img=img.rotate(rnd.uniform(-CFG['rotation_degrees'],CFG['rotation_degrees']),resample=Image.Resampling.BILINEAR)
            img=ImageEnhance.Brightness(img).enhance(rnd.uniform(1-CFG['brightness'],1+CFG['brightness']))
            img=ImageEnhance.Contrast(img).enhance(rnd.uniform(1-CFG['contrast'],1+CFG['contrast']))
        return self.normalize(transforms.functional.to_tensor(img)),int(self.s0['images'].loc[iid,'grade']),iid

class BalancedBatchSizes(Sampler):
    # Không cân bằng lớp: chỉ chia permutation thành batch; nhập mẫu dư=1 vào batch trước.
    def __init__(self,n,b,seed,shuffle):self.n=n;self.b=b;self.seed=seed;self.shuffle=shuffle
    def __iter__(self):
        order=np.random.default_rng(self.seed).permutation(self.n).tolist() if self.shuffle else list(range(self.n))
        batches=[order[i:i+self.b] for i in range(0,self.n,self.b)]
        if len(batches)>1 and len(batches[-1])==1:batches[-2]+=batches.pop()
        return iter(batches)
    def __len__(self):return math.ceil(self.n/self.b)-(self.n%self.b==1 and self.n>self.b)
def loader(ids,s0,paths,training=False,epoch=0):
    ds=FundusDataset(ids,s0,paths,training,epoch)
    gen=torch.Generator().manual_seed(RUN_SEED+epoch)
    return DataLoader(ds,batch_sampler=BalancedBatchSizes(len(ds),CFG['batch_size'],RUN_SEED+epoch,training),
        num_workers=CFG['num_workers'],pin_memory=torch.cuda.is_available(),generator=gen,persistent_workers=False)


## 7. ResNet-50 V2, GlobalAvgPool và checkpoint

Thay riêng avgpool bằng mean hai chiều không gian; giữ pretrained và classifier như bản gốc. `train_mode` khôi phục BN eval mỗi epoch nếu freeze. Checkpoint dang dở giữ model/optimizer/scheduler/scaler/RNG; checkpoint hoàn thành rút gọn để suy luận.


In [5]:
class GlobalAvgPool(nn.Module):
    def forward(self,x):return x.mean(dim=(2,3),keepdim=True)

def resolve_weights(root):
    """Lấy ĐÚNG ResNet50 ImageNet-1K V2: cache → Input sẵn có → tự tải PyTorch.

    SHA-256 filename prefix của weights torchvision được kiểm tra ngay cả khi file đã cache.
    Khi chưa có cache, Kaggle cần bật Internet ở lần chạy tải đầu tiên.
    """
    if not CFG['pretrained']:
        return None
    spec = models.ResNet50_Weights.IMAGENET1K_V2
    filename = 'resnet50-11ad3fa6.pth'
    expected_prefix = '11ad3fa6'
    require(spec.url.rsplit('/', 1)[-1] == filename,
            'URL ResNet50 IMAGENET1K_V2 khác phiên bản đã khóa; cần kiểm tra torchvision')
    cache_dir = Path(CFG['output_root']) / 'cache' / 'pretrained'
    cache_dir.mkdir(parents=True, exist_ok=True)
    destination = cache_dir / filename

    def verified(path):
        return path.is_file() and sha_file(path).startswith(expected_prefix)

    if verified(destination):
        LOG.info('Sử dụng ResNet50 ImageNet-1K V2 đã cache: %s', destination)
        return destination
    if destination.exists():
        LOG.warning('Cache pretrained sai SHA-256; xóa cache và lấy lại: %s', destination)
        destination.unlink()

    # Tái sử dụng file từ output Kaggle phiên trước (nếu đã attach để resume),
    # hoặc cache torchvision; luôn kiểm SHA và chỉ chọn bản V2.
    locations = [Path(torch.hub.get_dir()) / 'checkpoints' / filename]
    if Path(root).is_dir():
        locations.extend(Path(root).rglob(filename))
    for candidate in locations:
        if verified(candidate):
            shutil.copyfile(candidate, destination)
            require(verified(destination), 'File pretrained sai SHA sau khi copy vào cache')
            LOG.info('Đã phục hồi ResNet50 ImageNet-1K V2 từ cache/Input: %s', candidate)
            return destination

    LOG.info('Đang tự tải ResNet50 ImageNet-1K V2 từ PyTorch: %s', spec.url)
    # Tải vào file tạm; chỉ publish khi download và hash thành công.
    with tempfile.NamedTemporaryFile(dir=cache_dir, suffix='.download', delete=False) as f:
        temporary = Path(f.name)
    try:
        torch.hub.download_url_to_file(spec.url, str(temporary),
                                       hash_prefix=expected_prefix, progress=True)
        require(verified(temporary), 'SHA-256 của ResNet50 ImageNet-1K V2 không khớp')
        os.replace(temporary, destination)
    except Exception as exc:
        raise Blocked('Không tải được ResNet50 ImageNet-1K V2. '
                      'Hãy bật Internet trong Kaggle Settings rồi chạy lại; '
                      'hoặc attach cache V2 từ output phiên trước. '
                      f'Chi tiết: {type(exc).__name__}: {exc}') from exc
    finally:
        temporary.unlink(missing_ok=True)
    LOG.info('Đã tải và kiểm SHA-256 ResNet50 ImageNet-1K V2: %s', destination)
    return destination
def make_model(method,weights=None,tiny=False):
    if tiny:
        return nn.Sequential(nn.Flatten(),nn.Linear(3*8*8,12),nn.ReLU(),nn.Linear(12,K-1 if method=='CORN' else K))
    require(CFG['backbone']=='resnet50','Chỉ ResNet50 được triển khai/kiểm thử; không tự thay backbone')
    net=models.resnet50(weights=None)
    if weights is not None:
        state=torch.load(weights,map_location='cpu',weights_only=True)
        net.load_state_dict(state,strict=True)
    net.avgpool=GlobalAvgPool()
    # Seed riêng head tránh CORN làm thay đổi RNG cho backbone hay sampler.
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(RUN_SEED+137);net.fc=nn.Linear(net.fc.in_features,K-1 if method=='CORN' else K)
    for mod in net.modules():
        if isinstance(mod,nn.modules.batchnorm._BatchNorm):
            for p in mod.parameters():p.requires_grad_(not CFG['freeze_bn_affine'])
    return net
def bare(model):return model.module if isinstance(model,nn.DataParallel) else model
def scaler_new(enabled):return torch.amp.GradScaler('cuda',enabled=enabled)
def train_mode(model):
    model.train()
    if CFG['freeze_bn']:
        for mod in model.modules():
            if isinstance(mod,nn.modules.batchnorm._BatchNorm):mod.eval()
def checkpoint_state(model,opt,scheduler,scaler,epoch,best,bad,history,signature,best_epoch):
    return {'model':bare(model).state_dict(),'optimizer':opt.state_dict(),'scheduler':scheduler.state_dict(),
        'scaler':scaler.state_dict(),'epoch':epoch,'best_key':best,'bad_epochs':bad,'history':history,
        'signature':signature,'best_epoch':best_epoch,'rng':rng_state(),'protocol_hash':PROTOCOL_HASH}
def load_checkpoint(path,model,opt=None,scheduler=None,scaler=None,signature=None):
    # Chỉ nạp checkpoint do notebook này tạo từ output người dùng tin cậy; có RNG Python/NumPy.
    x=torch.load(path,map_location='cpu',weights_only=False)
    require(x['signature']==signature and x['protocol_hash']==PROTOCOL_HASH,'Checkpoint/config/hash không tương thích')
    bare(model).load_state_dict(x['model'])
    if opt is not None:
        opt.load_state_dict(x['optimizer']);scheduler.load_state_dict(x['scheduler']);scaler.load_state_dict(x['scaler']);restore_rng(x['rng'])
    return x
def check_time():
    hours=(time.monotonic()-START_TIME)/3600
    if hours>=CFG['session_hours']-CFG['stop_margin_minutes']/60:raise SessionLimit('Dừng chủ động để giữ output và resume phiên sau')
def compact_completed(run_dir,signature):
    # Sau commit completed: chỉ giữ model tốt nhất; optimizer/RNG chỉ cần cho run còn dang dở.
    p=run_dir/'best.pt'
    x=torch.load(p,map_location='cpu',weights_only=False)
    require(x['signature']==signature and x['protocol_hash']==PROTOCOL_HASH,'Completed checkpoint không khớp signature')
    if not x.get('inference_only'):
        keep={k:x[k] for k in ['model','signature','protocol_hash','epoch','best_epoch','best_key']}
        keep['inference_only']=True;atomic_torch(p,keep)
    (run_dir/'last.pt').unlink(missing_ok=True)


## 8. Công thức 10 phương pháp

$a_c=(1-\beta)/(1-\beta^{n_c})$; A3 dùng $w_c=Ka_c/\sum a_r$; các nhánh C dùng $w_c=K\sqrt{a_c}/\sum\sqrt{a_r}$, chỉ tính $n_c$ trên train fold.

$C=-w_y(1-p_y)^\gamma\log p_y$; $q_j=\sum_{c\le j}p_c$; $t_j=1[y\le j]$.

$O=-\frac1{K-1}\sum_j[t_j\log q_j+(1-t_j)\log(1-q_j)]$.

$e_j=|q_j-t_j|$, $N_{eff}=(\sum e_j)^2/\sum e_j^2$, $D=\frac12[1+(N_{eff}-1)/(K-2)]$.
Tính $N_{eff}$ từ e đã chia cho max(e) để ổn định; chỉ khi e bằng 0 thực sự mới đặt $N_{eff}=1$. Không lấy ngưỡng epsilon tùy ý để ép D=0,5 cho các sai số nhỏ nhưng khác 0.

| Mã | Loss | Đầu ra |
|---|---|---|
| A0 | CE | 5 logits |
| A3 | CB-Focal; β/γ tuned | 5 logits |
| A4 | C | 5 logits |
| A6 | CE + λO | 5 logits |
| A8 | C + λO, λ cố định sau tuning | 5 logits |
| A9 | C + λ mean((CDF−target)²) | 5 logits |
| A11 | O | 5 logits |
| A12 | C + sg(α wᵧ D)O | 5 logits |
| SLACE | SLACEprox, `max` theo repository | 5 logits |
| CORN | Conditional BCE / tổng số conditional examples | 4 logits |

RLS và squared EMD dùng trung bình K−1 ngưỡng; không thêm class weight vào O của A6/A8/A11. Không label smoothing. Tính log-CDF bằng logcumsumexp để tránh log(0).

In [6]:
def class_weights(counts,beta=0.999,sqrt=True,dtype=torch.float32):
    n=torch.as_tensor(counts,dtype=torch.float64)
    require(bool((n>0).all()),'Train fold thiếu class; không tự làm mượt count')
    require(0<=beta<1,'beta ngoài [0,1)')
    a=torch.ones_like(n) if beta==0 else (1-beta)/(-torch.expm1(n*math.log(beta)))
    w=a.sqrt() if sqrt else a
    return (len(n)*w/w.sum()).to(dtype)

def ordinal_parts(z,y):
    logp=F.log_softmax(z,dim=1)
    logq=torch.logcumsumexp(logp,1)[:,:-1]
    logr=torch.logcumsumexp(logp.flip(1),1).flip(1)[:,1:]
    t=y[:,None]<=torch.arange(z.shape[1]-1,device=z.device)[None,:]
    o=-torch.where(t,logq,logr).mean(1)
    # e = |CDF - t|, dùng xác suất phía sai tránh 1-exp(logq) bị cancellation.
    e=torch.where(t,logr.exp(),logq.exp())
    maximum=e.max(1,keepdim=True).values
    scaled=e/maximum.clamp_min(torch.finfo(e.dtype).tiny)
    neff=scaled.sum(1).square()/scaled.square().sum(1).clamp_min(torch.finfo(e.dtype).tiny)
    neff=torch.where(maximum[:,0]==0,torch.ones_like(neff),neff).clamp(1,z.shape[1]-1)
    d=(1+(neff-1)/(z.shape[1]-2))/2
    emd=e.square().mean(1)
    return logp,o,emd,d

def focal_vector(logp,y,w,gamma):
    lp=logp.gather(1,y[:,None]).squeeze(1)
    return -w[y]*(-torch.expm1(lp)).pow(gamma)*lp
def loss_A0(z,y,ctx):
    if CFG['a0_ce_backend']=='native':return F.cross_entropy(z,y,reduction='none'),{}
    require(CFG['a0_ce_backend']=='logsoftmax_gather','CE backend không hợp lệ')
    return -F.log_softmax(z,1).gather(1,y[:,None]).squeeze(1),{}
def loss_A3(z,y,ctx):return focal_vector(F.log_softmax(z,1),y,ctx.w,ctx.hp['gamma']),{}
def loss_A4(z,y,ctx):
    c=focal_vector(F.log_softmax(z,1),y,ctx.w,ctx.hp['gamma']);return c,{'C':c}
def loss_A6(z,y,ctx):
    lp,o,_,_=ordinal_parts(z,y);c=-lp.gather(1,y[:,None]).squeeze(1)
    return c+ctx.hp['lambda']*o,{'C':c,'O':o}
def loss_A8(z,y,ctx):
    lp,o,_,_=ordinal_parts(z,y);c=focal_vector(lp,y,ctx.w,ctx.hp['gamma'])
    return c+ctx.hp['lambda']*o,{'C':c,'O':o}
def loss_A9(z,y,ctx):
    lp,_,emd,_=ordinal_parts(z,y);c=focal_vector(lp,y,ctx.w,ctx.hp['gamma'])
    return c+ctx.hp['lambda']*emd,{'C':c,'O':emd}
def loss_A11(z,y,ctx):
    o=ordinal_parts(z,y)[1];return o,{'O':o}
def loss_A12(z,y,ctx):
    lp,o,_,d=ordinal_parts(z,y);c=focal_vector(lp,y,ctx.w,ctx.hp['gamma'])
    lam=(ctx.hp['alpha']*ctx.w[y]*d).detach()
    return c+lam*o,{'C':c,'O':o,'lambda':lam,'D':d.detach()}

def slace_tables(counts,alpha):
    n=torch.as_tensor(counts,dtype=torch.float64);k=len(n)
    require(bool((n>0).all()),'SLACE yêu cầu đủ K lớp ở train')
    prox=torch.empty(k,k,dtype=torch.float64)
    for predicted in range(k):
        for target in range(k):
            lo,hi=sorted([predicted,target]);mass=n[lo:hi+1].sum()-n[predicted]/2
            prox[predicted,target]=-torch.log(mass/n.sum())
    soft=F.softmax(alpha*prox.T,dim=1) # max(prox)-prox: hằng số triệt tiêu trong softmax
    dominance=[]
    for target in range(k):
        order=prox[:,target].argsort().flip(0)
        rank=torch.empty(k,dtype=torch.long);rank[order]=torch.arange(k)
        dominance.append(rank[None,:]<=rank[:,None])
    return soft,torch.stack(dominance)
def loss_SLACE(z,y,ctx):
    lp=F.log_softmax(z,1)
    logmass=torch.logsumexp(lp[:,None,:].masked_fill(~ctx.dom[y],-torch.inf),dim=2)
    v=-(ctx.soft[y]*logmass).sum(1)
    return v,{}
def corn_sums(z,y):
    j=torch.arange(z.shape[1],device=z.device)
    eligible=y[:,None]>=j;target=(y[:,None]>j).to(z.dtype)
    values=F.binary_cross_entropy_with_logits(z,target,reduction='none')
    return (values*eligible).sum(),eligible.sum()
def loss_CORN(z,y,ctx):
    num,den=corn_sums(z,y);return num/den,{}
def corn_decode(z):
    require(z.ndim==2 and z.shape[1]==K-1,'CORN cần conditional logits (N,K-1)')
    return (torch.sigmoid(z).cumprod(dim=1)>.5).sum(dim=1)

def decoded_predictions(p,method=None):
    p=np.asarray(p)
    # Sum the tail directly: avoids cancellation from 1-CDF near 0.5.
    return (np.cumsum(p[:,::-1],axis=1)[:,::-1][:,1:]>.5).sum(1) if method=='CORN' else p.argmax(1)

def probabilities(z,method):
    if method!='CORN':return F.softmax(z,1)
    survival=z.sigmoid().cumprod(1)
    return torch.cat([1-survival[:,:1],survival[:,:-1]-survival[:,1:],survival[:,-1:]],1)
LOSS_FUNCTIONS={m:globals()['loss_'+m] for m in METHODS}
class LossContext(nn.Module):
    def __init__(self,method,counts,hp):
        super().__init__();self.method=method;self.hp=hp
        self.register_buffer('w',class_weights(counts,hp.get('beta',CFG['sqrt_beta']),method!='A3'))
        soft,dom=slace_tables(counts,hp.get('alpha',1.0))
        self.register_buffer('soft',soft.float());self.register_buffer('dom',dom)
    def forward(self,z,y):
        # FP16 model -> FP32 loss; preserve float64 for numerical tests.
        z=z.float() if z.dtype in (torch.float16,torch.bfloat16) else z
        value,parts=LOSS_FUNCTIONS[self.method](z,y,self)
        return value.mean(),{k:v.mean() for k,v in parts.items()}
    def numerator_denominator(self,z,y):
        z=z.float() if z.dtype in (torch.float16,torch.bfloat16) else z
        if self.method=='CORN':
            n,d=corn_sums(z,y);return n,d,{}
        v,c=LOSS_FUNCTIONS[self.method](z,y,self)
        return v.sum(),len(y),{k:x.sum() for k,x in c.items()}

def parameter_grid(method):
    base={'beta':CFG['sqrt_beta'],'gamma':CFG['sqrt_gamma']}
    if method=='A3':return [{'beta':b,'gamma':g} for b in CFG['beta_grid_A3'] for g in CFG['gamma_grid_A3']]
    if method in ('A6','A8','A9'):
        grid=CFG['lambda_grid_A8'] if method=='A8' else CFG['lambda_grid_A6_A9']
        return [dict(base,**{'lambda':v}) for v in grid]
    if method=='A12':return [dict(base,alpha=a) for a in CFG['alpha_grid_A12']]
    if method=='SLACE':return [{'alpha':a,'variant':'prox_max'} for a in CFG['slace_alpha_grid']]
    return [base if method=='A4' else {}]


## 9. SLACE/CORN — nguồn và phần điều chỉnh

**SLACE:** Inbar Nachmani, Bar Genossar, Coral Scharf, Roee Shraga, Avigdor Gal (AAAI 2025), *SLACE: A Monotone and Balance-Sensitive Loss Function for Ordinal Regression*. [Bài báo](https://ojs.aaai.org/index.php/AAAI/article/view/34158), [full paper từ tác giả](https://github.com/inbarnachmani/SLACE/blob/main/SLACE_full_paper.pdf), [losses.py](https://github.com/inbarnachmani/SLACE/blob/main/losses.py), [utils.py](https://github.com/inbarnachmani/SLACE/blob/main/utils.py).

Triển khai `SLACE_max`/`accumulating_with_sord_prox(type='max')`, là bản proximity dùng class counts, không phải bản `SLACE` khoảng cách tuyệt đối đơn thuần trong repository. Với $R_{cy}=prox(c,y)$ theo Eq.7, soft target $s_{yc}=softmax_c(\alpha R_{cy})$, accumulation theo thứ tự proximity như `create_prox_dom` của tác giả:

$L_i=-\sum_c s_{y_i c}\log\sum_{j\in A(y_i,c)}p_{ij}$.

Giữ nguyên proximity/soft targets/accumulation. Mã gốc chạy XGBoost trên dữ liệu bảng, nhận mảng xác suất, detach để tạo tensor mới, tự gọi backward và trả gradient cùng Hessian giả bằng 1. **Bản CNN** giữ graph từ logits, tính logsumexp trên các class được tích lũy, trả loss trung bình theo mẫu thay cho tổng toàn tập; không đưa Hessian giả vào AdamW. Counts được cố định từ toàn train fold, không tính lại theo minibatch. Tests đối chiếu tổng loss và gradient theo xác suất, sau đó chain rule qua softmax.

Thứ tự khi proximity bằng nhau theo `argsort` của repository; công thức tập hợp trong bài có thể xử lý hòa khác. Giữ hành vi repository và ghi rõ giới hạn này. Các định lý cho xác suất/loss không tự chứng minh kết quả học CNN. Chưa chạy thực nghiệm CNN APTOS trong môi trường tạo file.

**CORN:** Xintong Shi, Wenzhi Cao, Sebastian Raschka (2023; preprint 2021), *Deep Neural Networks for Rank-Consistent Ordinal Regression Based On Conditional Probabilities*. [Paper](https://arxiv.org/abs/2111.08851), [code thí nghiệm](https://github.com/Raschka-research-group/corn-ordinal-neuralnet), [hàm tham chiếu được pin](https://github.com/Raschka-research-group/coral-pytorch/blob/313482f86f50b58d8beb9fb54652e943b06745ef/coral_pytorch/losses.py).

Head K−1 không chia sẻ trọng số bắt buộc. Task j chỉ nhận các mẫu $y\ge j$ và target $1[y>j]$. Loss chia tổng BCE cho tổng số phần tử của các conditional subsets. $u_j=\prod_{r=0}^{j}\sigma(z_r)$; $p_0=1-u_0$, $p_c=u_{c-1}-u_c$, $p_{K-1}=u_{K-2}$. Dùng median decoder $\sum_j1[u_j>0.5]$ trong validation, chọn checkpoint và bảng chính; argmax chỉ là kết quả phụ. Điều chỉnh backbone sang CNN dùng chung là tương thích thiết kế CORN.

Hàm tham chiếu bên dưới được trích trước vào notebook để test offline. Hash bytes nguồn được ghi trong REFERENCE_SHA256. Không tải hoặc thực thi mã từ GitHub khi chạy Kaggle. Không dùng nhánh `main` thay cho commit CORN đã kiểm chứng.

In [7]:
# Các hàm tham chiếu trích từ mã tác giả; không tải mạng khi chạy Kaggle.
from collections import Counter
f = F
device = "cpu"

def reference_create_prox_mat(dist_dict, inv=True):
    labels = list(dist_dict.keys())
    labels.sort()
    denominator = sum(dist_dict.values())
    prox_mat = np.zeros([len(labels), len(labels)])
    for label1 in labels:
        for label2 in labels:
            label1 = int(label1)
            label2 = int(label2)
            minlabel, maxlabel = min(label1, label2), max(label1, label2)
            numerator = dist_dict[label1] / 2
            if minlabel == label1:  # Above the diagonal
                for tmp_label in range(minlabel + 1, maxlabel + 1):
                    numerator += dist_dict[tmp_label]
            else:  # Under the diagonal
                for tmp_label in range(maxlabel - 1, minlabel - 1, -1):
                    numerator += dist_dict[tmp_label]
            if inv:
                prox_mat[label1 - 1][label2 - 1] = (-np.log(numerator / denominator)) ** -1
            else:
                prox_mat[label1 - 1][label2 - 1] = -np.log(numerator / denominator)
    return torch.tensor(prox_mat)

def reference_create_prox_dom(prox_mat):
    labels_num = prox_mat.shape[0]
    prox_dom_mat = []
    for label in range(prox_mat.shape[0]):
        label_prox_ordered = reversed(prox_mat[:, label].argsort()).tolist()
        prox_dom_mat.append(torch.tensor([[0.0 if label_prox_ordered.index(ind_axe) <
                                                  label_prox_ordered.index(ind_inner) else 1.0
                                           for ind_inner in range(labels_num)]
                                          for ind_axe in range(labels_num)]))
    return torch.stack(prox_dom_mat)

def reference_change_inputs(targets, output_vals):
    targets = targets + 1
    labels_num = int(np.max(targets))
    d = Counter(targets)
    dist_dict = {int(k): int(v) for k, v in d.items()}
    prox_mat = reference_create_prox_mat(dist_dict, inv=False)
    norm_prox_mat = f.normalize(prox_mat, p=1, dim=0)
    prox_dom = reference_create_prox_dom(prox_mat).double()

    targets = targets - 1
    output_vals = torch.tensor(output_vals, requires_grad=True)
    targets = torch.tensor(targets)
    targets.requires_grad_(False)
    num_examples = targets.shape[0]
    # print(norm_prox_mat)
    # print(prox_mat)
    return labels_num, prox_mat.T, norm_prox_mat.T, prox_dom, output_vals, targets.long(), num_examples

def reference_accumulating_with_sord_prox(alpha, return_loss=False, type="max"):
    def Loss(targets, output_vals):
        labels_num, prox_mat, norm_prox_mat, prox_dom, output_vals, targets, num_examples = reference_change_inputs(targets,
                                                                                                           output_vals)

        if type == "max":
            phi = torch.max(prox_mat) - prox_mat[targets].to(device)
        if type == "norm_max":
            phi = torch.max(norm_prox_mat) - norm_prox_mat[targets].to(device)
        if type == "norm_log":
            phi = - torch.log(norm_prox_mat[targets].to(device))
        if type == "log":
            phi = - torch.log(prox_mat[targets].to(device))
        if type == "norm_division":
            phi = 1/(norm_prox_mat[targets].to(device))
        if type == "division":
            phi = 1/(prox_mat[targets].to(device))

        softmax_targets = f.softmax(-alpha * phi, dim=1).to(device)

        one_hot_target = torch.tensor(np.zeros([num_examples, labels_num])).to(device)
        one_hot_target[range(num_examples), targets] = 1
        one_hot_target_comp = 1 - one_hot_target
        mass_weights = one_hot_target * softmax_targets + one_hot_target_comp * softmax_targets

        accumulating_softmax = torch.matmul(prox_dom[targets.long()].double(),
                                            torch.unsqueeze(output_vals, 2).double()).double().squeeze(dim=2)

        loss_val = -1 * torch.sum(mass_weights * torch.log(accumulating_softmax))

        if return_loss == True:
            return loss_val

        loss_val.backward()
        grads = output_vals.grad.numpy()

        del output_vals, targets, accumulating_softmax, one_hot_target, one_hot_target_comp, \
            mass_weights

        return grads.flatten(), np.ones(grads.shape).flatten()

    return Loss

def reference_corn_loss(logits, y_train, num_classes):
    """Computes the CORN loss described in our forthcoming
    'Deep Neural Networks for Rank Consistent Ordinal
    Regression based on Conditional Probabilities'
    manuscript.

    Parameters
    ----------
    logits : torch.tensor, shape=(num_examples, num_classes-1)
        Outputs of the CORN layer.

    y_train : torch.tensor, shape=(num_examples)
        Torch tensor containing the class labels.

    num_classes : int
        Number of unique class labels (class labels should start at 0).

    Returns
    ----------
        loss : torch.tensor
        A torch.tensor containing a single loss value.

    Examples
    ----------
    >>> import torch
    >>> from coral_pytorch.losses import reference_corn_loss
    >>> # Consider 8 training examples
    >>> _  = torch.manual_seed(123)
    >>> X_train = torch.rand(8, 99)
    >>> y_train = torch.tensor([0, 1, 2, 2, 2, 3, 4, 4])
    >>> NUM_CLASSES = 5
    >>> #
    >>> #
    >>> # def __init__(self):
    >>> corn_net = torch.nn.Linear(99, NUM_CLASSES-1)
    >>> #
    >>> #
    >>> # def forward(self, X_train):
    >>> logits = corn_net(X_train)
    >>> logits.shape
    torch.Size([8, 4])
    >>> reference_corn_loss(logits, y_train, NUM_CLASSES)
    tensor(0.7127, grad_fn=<DivBackward0>)
    """
    sets = []
    for i in range(num_classes-1):
        label_mask = y_train > i-1
        label_tensor = (y_train[label_mask] > i).to(torch.int64)
        sets.append((label_mask, label_tensor))

    num_examples = 0
    losses = 0.
    for task_index, s in enumerate(sets):
        train_examples = s[0]
        train_labels = s[1]

        if len(train_labels) < 1:
            continue

        num_examples += len(train_labels)
        pred = logits[train_examples, task_index]

        loss = -torch.sum(F.logsigmoid(pred)*train_labels
                          + (F.logsigmoid(pred) - pred)*(1-train_labels))
        losses += loss

    return losses/num_examples

REFERENCE_SHA256 = {'slace_utils.py': '5dd00a192c7e24089b64de34905cbb7bd9e5903c1cced8854abbda1a091d106e', 'slace.py': 'f55172d89ad8b3805b2db3082825a3e93981b1d7dc117601f21e34ea08ce0657', 'corn_pinned.py': '6c182fb00675639392336f70d7a9ae72a2ef8cfddff2889f2ea765fce539c6fa'}


## 10. Metrics, temperature scaling và bootstrap

QWK cố định 5 lớp, Accuracy, Balanced Accuracy, Macro/Weighted-F1, macro precision/recall, per-class, MAE/RMSE, lỗi ≥2 grade, recall grade 3/4, NLL, ECE 15 bins, RPS và Brier. QWK không xác định được ghi null. Macro-F1 lấy đủ 5 lớp, absent class có F1=0; recall 3/4 absent được ghi null.

Tối ưu temperature T trên validation NLL (T=exp(logT), logT∈[−4,4]); áp dụng lên phân phối 5 lớp cho cả CORN. Không dùng test để chọn T. So sánh raw/scaled dùng cùng checkpoint.

Paired bootstrap 2.000 lần trên cùng ảnh theo QĐ19, thêm bootstrap theo S0 cluster; CI điều kiện theo các mô hình đã fit, không bao gồm bất định huấn luyện lại/tuning. Test được suy luận sau selection lock, không sau từng epoch. Nếu gián đoạn đúng sau suy luận nhưng trước commit, một test inference tất định có thể chạy lại để phục hồi artifact; không dùng kết quả đó để chọn model.

In [8]:
def metric_from_cm(cm):
    cm=np.asarray(cm,dtype=float);n=cm.sum()
    if n==0:return {'n':0,'QWK':None,'Macro_F1':None,'Accuracy':None,'MAE':None,'far_error':None}
    true=cm.sum(1);pred=cm.sum(0);tp=cm.diagonal();idx=np.arange(K)
    dist=np.abs(idx[:,None]-idx[None,:]);weights=dist**2
    expected=np.outer(true,pred)/n;den=(weights*expected).sum()
    precision=np.divide(tp,pred,out=np.zeros(K),where=pred>0)
    recall=np.divide(tp,true,out=np.zeros(K),where=true>0)
    f1=np.divide(2*precision*recall,precision+recall,out=np.zeros(K),where=precision+recall>0)
    return {'n':int(n),'QWK':float(1-(weights*cm).sum()/den) if den>0 else None,
        'Accuracy':float(tp.sum()/n),'Balanced_Accuracy':float(recall[true>0].mean()),
        'Macro_F1':float(f1.mean()),'Weighted_F1':float((f1*true).sum()/n),
        'Macro_Precision':float(precision.mean()),'Macro_Recall':float(recall.mean()),
        'MAE':float((dist*cm).sum()/n),'RMSE':float(np.sqrt((weights*cm).sum()/n)),
        'far_error':float(cm[dist>=2].sum()/n),'Recall_3':float(recall[3]) if true[3]>0 else None,
        'Recall_4':float(recall[4]) if true[4]>0 else None,
        'confusion_matrix':cm.astype(int).tolist(),
        'confusion_matrix_row_normalized':np.divide(cm,true[:,None],out=np.zeros_like(cm),where=true[:,None]>0).tolist(),
        'per_class':{str(i):{'precision':float(precision[i]),'recall':float(recall[i]),'f1':float(f1[i]),'support':int(true[i])} for i in range(K)}}
def metrics(y,p,pred=None,method=None):
    y=np.asarray(y,dtype=int);p=np.asarray(p,dtype=np.float64)
    require(p.shape==(len(y),K) and np.isfinite(p).all(),'Probability shape/finite sai')
    require((p>=-1e-8).all() and np.allclose(p.sum(1),1,atol=1e-5),'Probability không hợp lệ')
    pred=decoded_predictions(p,method) if pred is None else np.asarray(pred,dtype=int)
    require(pred.shape==(len(y),) and ((pred>=0)&(pred<K)).all(),'Prediction shape/range sai')
    cm=np.bincount(y*K+pred,minlength=K*K).reshape(K,K)
    result=metric_from_cm(cm)
    if not len(y):return result
    conf=p[np.arange(len(y)),pred];correct=pred==y;bin_id=np.minimum((conf*15).astype(int),14)
    ece=0;bins=[]
    for b in range(15):
        ix=bin_id==b;count=int(ix.sum())
        acc=float(correct[ix].mean()) if count else None;confidence=float(conf[ix].mean()) if count else None
        if count:ece+=count/len(y)*abs(acc-confidence)
        bins.append({'bin':b,'n':count,'accuracy':acc,'confidence':confidence})
    target=y[:,None]<=np.arange(K-1)
    result.update({'NLL':float(-np.log(p[np.arange(len(y)),y].clip(1e-15)).mean()),
        'ECE15':float(ece),'RPS':float(((p.cumsum(1)[:,:-1]-target)**2).mean()),
        'Brier':float(((p-np.eye(K)[y])**2).sum(1).mean()),'calibration_bins':bins})
    return result
def scaled_probs(p,t):
    lp=np.log(np.asarray(p,dtype=float).clip(1e-15))/t;lp-=lp.max(1,keepdims=True)
    exp=np.exp(lp);return exp/exp.sum(1,keepdims=True)
def fit_temperature(y,p):
    result=minimize_scalar(lambda logt:metrics(y,scaled_probs(p,np.exp(logt)))['NLL'],bounds=(-4,4),method='bounded')
    require(result.success and np.isfinite(result.fun),'Temperature fit failed')
    t=float(np.exp(result.x))
    # T=1 thuộc miền tìm kiếm; không chấp nhận nghiệm tệ hơn gốc.
    return t if result.fun<=metrics(y,p)['NLL'] else 1.0
def confidence_interval(vals):
    vals=[x for x in vals if x is not None and np.isfinite(x)]
    return {'lower':float(np.quantile(vals,.025)) if vals else None,'upper':float(np.quantile(vals,.975)) if vals else None,'valid_replicates':len(vals)}
def paired_bootstrap(y,pa,pb,groups=None,reps=None,method_a=None,method_b=None,pred_a=None,pred_b=None):
    require(len(y)==len(pa)==len(pb),'Paired length sai')
    if not len(y):return {'status':'empty_subset'}
    reps=CFG['bootstrap_reps'] if reps is None else reps
    rng=np.random.default_rng(CFG['bootstrap_seed']);a=decoded_predictions(pa,method_a) if pred_a is None else np.asarray(pred_a);b=decoded_predictions(pb,method_b) if pred_b is None else np.asarray(pred_b)
    y=np.asarray(y);sample_groups=None
    if groups is not None:
        sample_groups=[np.flatnonzero(np.asarray(groups)==g) for g in np.unique(groups)]
    vals={k:[] for k in ['QWK','Macro_F1','MAE','far_error']}
    for _ in range(reps):
        ix=rng.integers(0,len(y),len(y)) if sample_groups is None else np.concatenate([sample_groups[g] for g in rng.integers(0,len(sample_groups),len(sample_groups))])
        ma=metric_from_cm(np.bincount(y[ix]*K+a[ix],minlength=K*K).reshape(K,K))
        mb=metric_from_cm(np.bincount(y[ix]*K+b[ix],minlength=K*K).reshape(K,K))
        for k in vals:vals[k].append(None if ma[k] is None or mb[k] is None else ma[k]-mb[k])
    ra=metrics(y,pa,pred=a);rb=metrics(y,pb,pred=b)
    return {'status':'computed','unit':'image' if groups is None else 's0_cluster','reps':reps,'seed':CFG['bootstrap_seed'],
        'n':len(y),'differences':{k:{'point':None if ra[k] is None or rb[k] is None else ra[k]-rb[k],**confidence_interval(v)} for k,v in vals.items()},
        'limitation':'conditional on fitted fold models; does not include refit or tuning uncertainty'}

def subgroup_masks(s0,ids):
    ac=s0['acquisition'].loc[ids];is_p=ac.size_group.isin(['1050x1050','2048x1536','819x614']).to_numpy()
    masks={'all':np.ones(len(ids),dtype=bool),'P':is_p,'M':~is_p}
    for group in sorted(ac.acquisition_group.unique()):masks['acquisition_'+str(group)]=(ac.acquisition_group==group).to_numpy()
    sm=s0['masks'].set_index('id_code').loc[ids]
    for col in ['unique_pixel_test_keep','phash4_filtered_test_keep']:masks[col]=sm[col].to_numpy(dtype=bool)
    val0=set(next(f for f in s0['folds'] if f['fold']==0)['validation'])
    masks['exclude_fold0_validation']=np.array([i not in val0 for i in ids],dtype=bool)
    if 'crosscheck_test_keep' in sm:
        masks['crosscheck_test_keep']=sm['crosscheck_test_keep'].to_numpy(dtype=bool)
    return masks
def validate_oof(s0,records):
    seen=set()
    for f,arr in records:
        ids=arr['ids'].astype(str).tolist()
        require(ids==list(f['test']),'OOF thứ tự/ID không khớp test fold')
        require(not seen&set(ids),'OOF ID lặp');seen.update(ids)
        require(np.array_equal(arr['y'],s0['images'].loc[ids].grade),'OOF nhãn sai')
        require(len(arr['p'])==len(ids),'OOF probability length sai')
    require(seen==s0['eligible'],'OOF chưa đủ dữ liệu')



## 11. Unit tests của loss, gradient và JSON

Kiểm tra giá trị cực đoan, gradcheck, sqrt normalization, A8 fixed λ, A12 stop-gradient, RLS/EMD, SLACE/CORN so mã tác giả, tổng xác suất, denominator khi tích lũy gradient và writer/privacy. Không gradcheck trực tiếp toàn hàm A12 có hệ số detach bằng sai phân hữu hạn; kiểm tra gradient với hệ số đóng băng và đối chứng bỏ stop-gradient.

In [9]:
def run_unit_tests():
    results={};seed_all(123);counts=[7,5,3,2,1]
    def passed(name):results[name]='passed'
    # K lớp đầy đủ để implementation tác giả xây được proximity matrix, counts có mất cân bằng.
    y=torch.repeat_interleave(torch.arange(K),torch.tensor(counts))
    z=torch.randn(len(y),K,dtype=torch.float64,requires_grad=True)
    for method in METHODS:
        hp=parameter_grid(method)[0];ctx=LossContext(method,counts,hp).double()
        logits=z[:,:K-1].detach().clone().requires_grad_(True) if method=='CORN' else z.clone()
        val,_=ctx(logits,y);g=torch.autograd.grad(val,logits)[0]
        require(torch.isfinite(val) and torch.isfinite(g).all(),method+' finite forward/backward failed')
        extreme=(torch.arange(len(y)*(K-1 if method=='CORN' else K),dtype=torch.float64).reshape(len(y),-1)%3-1)*1000
        extreme.requires_grad_();v,_=ctx(extreme,y);gg=torch.autograd.grad(v,extreme)[0]
        require(torch.isfinite(v) and torch.isfinite(gg).all(),method+' extreme logits failed')
        passed(method+'_finite_extreme')
    w=class_weights(counts,.999,True,dtype=torch.float64)
    a=(1-.999)/(1-np.power(.999,np.asarray(counts)))
    require(np.allclose(w.numpy(),K*np.sqrt(a)/np.sqrt(a).sum()) and abs(float(w.mean())-1)<1e-12,'sqrt normalization sai');passed('sqrt_normalization')
    lp,o,e,d=ordinal_parts(z,y);cdf=lp.exp().cumsum(1)[:,:-1];t=(y[:,None]<=torch.arange(K-1)).double()
    require(torch.allclose(o,F.binary_cross_entropy(cdf,t,reduction='none').mean(1)),'RLS sai')
    require(torch.allclose(e,((cdf-t)**2).mean(1)),'EMD sai');passed('RLS_EMD_reference')
    ctx=LossContext('A8',counts,parameter_grid('A8')[0]).double()
    v,_=ctx(z,y);expected=(focal_vector(lp,y,ctx.w,ctx.hp['gamma'])+ctx.hp['lambda']*o).mean()
    require(torch.allclose(v,expected),'A8 fixed lambda sai');passed('A8_fixed_lambda')
    ctx=LossContext('A12',counts,parameter_grid('A12')[0]).double();v,_=ctx(z,y)
    expected=(focal_vector(lp,y,ctx.w,ctx.hp['gamma'])+(ctx.hp['alpha']*ctx.w[y]*d).detach()*o).mean()
    actual_g=torch.autograd.grad(v,z,retain_graph=True)[0];expected_g=torch.autograd.grad(expected,z,retain_graph=True)[0]
    require(torch.allclose(actual_g,expected_g),'A12 stop-gradient sai')
    no_sg=(focal_vector(lp,y,ctx.w,ctx.hp['gamma'])+ctx.hp['alpha']*ctx.w[y]*d*o).mean()
    require(not torch.allclose(actual_g,torch.autograd.grad(no_sg,z,retain_graph=True)[0]),'Test SG không phân biệt được');passed('A12_stop_gradient')
    for method in ['A0','A3','A4','A6','A8','A9','A11','SLACE','CORN']:
        ctx=LossContext(method,counts,parameter_grid(method)[0]).double()
        zz=torch.randn(3,K-1 if method=='CORN' else K,dtype=torch.float64,requires_grad=True)
        yy=torch.tensor([0,2,4])
        require(torch.autograd.gradcheck(lambda x:ctx(x,yy)[0],(zz,),eps=1e-6,atol=1e-4),'GradCheck '+method)
    passed('GradCheck_9_losses_A12_checked_with_frozen_coefficient')
    for alpha in CFG['slace_alpha_grid']:
        # Reference receives probabilities, returns sum and gradient w.r.t probabilities.
        probs=z.detach().softmax(1)
        ref=reference_accumulating_with_sord_prox(alpha,return_loss=True,type='max')(y.numpy(),probs.numpy())
        refgrad,_=reference_accumulating_with_sord_prox(alpha,return_loss=False,type='max')(y.numpy(),probs.numpy())
        ctx=LossContext('SLACE',counts,{'alpha':alpha}).double();v,_=ctx(z,y)
        require(torch.allclose(v*len(y),ref,atol=2e-6,rtol=2e-7),'SLACE reference loss sai')
        gp=torch.as_tensor(refgrad.reshape(len(y),K))/len(y)
        chain=probs*(gp-(gp*probs).sum(1,keepdim=True))
        got=torch.autograd.grad(v,z,retain_graph=True)[0]
        require(torch.allclose(got,chain,atol=2e-7,rtol=2e-6),'SLACE reference gradient sai')
    passed('SLACE_author_loss_gradient_all_grid')
    for yy in [y,torch.zeros(6,dtype=torch.long),torch.full((6,),4,dtype=torch.long)]:
        zz=torch.randn(len(yy),K-1,dtype=torch.float64,requires_grad=True)
        a=reference_corn_loss(zz,yy,K);n,den=corn_sums(zz,yy);b=n/den
        require(torch.allclose(a,b),'CORN reference loss sai')
        ga=torch.autograd.grad(a,zz,retain_graph=True)[0];gb=torch.autograd.grad(b,zz)[0]
        require(torch.allclose(ga,gb),'CORN reference gradient sai')
        p=probabilities(zz,'CORN');require((p>=0).all() and torch.allclose(p.sum(1),torch.ones(len(yy),dtype=p.dtype)),'CORN probability sai')
        survival=zz.sigmoid().cumprod(1)
        require(torch.allclose(1-p.cumsum(1)[:,:-1],survival),'CORN chain rule sai')
    passed('CORN_pinned_author_forward_gradient_probability')
    for m in ['A8','CORN']:
        yy=torch.tensor([0,0,1,4,2,3,0]);zz=torch.randn(7,K-1 if m=='CORN' else K,dtype=torch.float64,requires_grad=True)
        ctx=LossContext(m,counts,parameter_grid(m)[0]).double();n,den,_=ctx.numerator_denominator(zz,yy)
        pieces=[ctx.numerator_denominator(zz[:3],yy[:3]),ctx.numerator_denominator(zz[3:],yy[3:])]
        require(torch.allclose(n/den,sum(p[0] for p in pieces)/sum(p[1] for p in pieces)),'Accumulation denominator sai')
    passed('weighted_accumulation_including_CORN')
    yy=np.array([0,1,2,3,4,1,4]);pp=np.eye(K)[[0,2,2,4,4,1,3]]*.98+.02/K;r=metrics(yy,pp)
    require(abs(r['QWK']-cohen_kappa_score(yy,pp.argmax(1),labels=list(range(K)),weights='quadratic'))<1e-12,'QWK sai')
    require(abs(r['Macro_F1']-precision_recall_fscore_support(yy,pp.argmax(1),labels=list(range(K)),zero_division=0)[2].mean())<1e-12,'Macro F1 sai')
    t=fit_temperature(yy,pp);require(metrics(yy,scaled_probs(pp,t))['NLL']<=r['NLL']+1e-8,'Temperature worse')
    bc=paired_bootstrap(yy,pp,pp,reps=30);require(bc['differences']['QWK']['point']==0,'Paired bootstrap sai');passed('metrics_calibration_bootstrap')
    for obj in [{'id_code':['abc']},{'safe':'/kaggle/input/a.jpg'},{'safe':['abcdef123456.png']}]:
        try:metric_privacy_check(obj)
        except Blocked:pass
        else:raise AssertionError('Privacy test failed')
    with tempfile.TemporaryDirectory() as tmp:
        p=Path(tmp)/'metric.json';atomic_json(p,{'a':float('nan'),'b':float('inf'),'c':[[1,2],[3,4]]})
        require(read_json(p)['a'] is None,'NaN cleaning failed')
    passed('metric_privacy_atomic_finite')
    return results


def run_fix_tests():
    global OUT,RUN_SEED
    result={};saved=copy.deepcopy(CFG);old_out=globals().get('OUT');old_seed=RUN_SEED
    try:
        seed_all(42)
        x=torch.randn(4,2048,12,12,requires_grad=True)
        a=nn.AdaptiveAvgPool2d(1)(x);b=GlobalAvgPool()(x)
        require(a.shape==b.shape and (a-b).abs().max().item()<=1e-6,'Pooling equivalence failed')
        ga=torch.autograd.grad(a.sum(),x,retain_graph=True)[0];gb=torch.autograd.grad(b.sum(),x)[0]
        require(torch.equal(ga,gb),'Pooling gradient khác')
        result['global_pool_cpu_output_gradient']='passed'
        for frozen in (True,False):
            CFG['freeze_bn']=frozen;bn=nn.BatchNorm2d(3);net=nn.Sequential(bn,nn.Conv2d(3,3,1))
            before=bn.running_mean.clone();train_mode(net);net(torch.ones(4,3,8,8)*2).sum().backward()
            require(torch.equal(before,bn.running_mean)==frozen,'BN running stats sai')
            require(bn.weight.grad is not None and bn.weight.requires_grad,'BN affine bị freeze ngoài ý muốn')
            net.eval();train_mode(net);require(bn.training is (not frozen),'train_mode không duy trì freeze')
        result['freeze_bn_stats_affine']='passed'
        zz=torch.randn(12,K,dtype=torch.float64,requires_grad=True);yy=torch.arange(12)%K
        a=F.cross_entropy(zz,yy,reduction='none');b=-F.log_softmax(zz,1).gather(1,yy[:,None]).squeeze(1)
        require(torch.allclose(a,b,atol=1e-12,rtol=0),'CE fallback value sai')
        require(torch.allclose(torch.autograd.grad(a.sum(),zz,retain_graph=True)[0],torch.autograd.grad(b.sum(),zz)[0],atol=1e-12,rtol=0),'CE fallback gradient sai')
        result['ce_fallback_equivalence']='passed'
        conditional=torch.tensor([[.9,.8/.9,.3/.8,.1/.3],[.1,.5,.5,.5],[.99,.99,.99,.99],[.5,1.,1.,1.]],dtype=torch.float64)
        zz=torch.logit(conditional);pred=corn_decode(zz)
        require(pred.tolist()==[2,0,4,0],'CORN hand/batch/tie test failed')
        pp=probabilities(zz,'CORN').numpy()
        require(np.array_equal(decoded_predictions(pp,'CORN'),pred.numpy()),'CORN probability decoder khác logits')
        # Deliberately use a distribution whose mode differs from ordinal median.
        pp=np.array([[.4,.2,.15,.15,.1]])
        require(decoded_predictions(pp,'CORN').item()==1 and pp.argmax(1).item()==0,'CORN test không phân biệt argmax')
        r=metrics([1],pp,method='CORN');require(r['Accuracy']==1 and np.isfinite(r['NLL']) and np.isfinite(r['ECE15']),'CORN metrics không dùng decoder')
        require(abs(r['ECE15']-.8)<1e-12,'ECE confidence phải p của nhãn được dự đoán')
        result['corn_decoder_metrics']='passed'
        with tempfile.TemporaryDirectory() as tmp:
            OUT=Path(tmp);RUN_SEED=42;CFG.update({'input_size':32,'num_workers':0,'cache_images':True})
            ar=np.arange(41*53*3,dtype=np.uint16).reshape(41,53,3)%220+10
            path=OUT/'fixture.png';Image.fromarray(ar.astype(np.uint8)).save(path)
            fake={'images':pd.DataFrame({'grade':[2]},index=['fixture']), 'summary':{'dataset_fingerprint':'synthetic_fixture'}}
            ds=FundusDataset(['fixture'],fake,{'fixture':path},True,3)
            a=ds[0][0];b=ds[0][0]
            require(torch.equal(a,b),'Augmentation không deterministic')
            for method in METHODS:
                seed_all(100+METHODS.index(method))
                require(torch.equal(a,FundusDataset(['fixture'],fake,{'fixture':path},True,3)[0][0]),'Augmentation phụ thuộc method/global RNG')
            a=FundusDataset(['fixture'],fake,{'fixture':path},False,0)[0][0]
            b=FundusDataset(['fixture'],fake,{'fixture':path},False,99)[0][0]
            require(torch.equal(a,b),'Validation/test có random augmentation')
            for epoch in range(100):
                rnd=random.Random(int(hashlib.sha256(f'{RUN_SEED}:{epoch}:fixture'.encode()).hexdigest()[:16],16))
                rnd.random();rnd.random();rnd.uniform(-CFG['rotation_degrees'],CFG['rotation_degrees'])
                factors=[rnd.uniform(1-CFG[k],1+CFG[k]) for k in ('brightness','contrast')]
                require(all(.9<=f<=1.1 for f in factors),'Augmentation factors ngoài miền')
            # Verify actual ordering with an independent implementation.
            ds=FundusDataset(['fixture'],fake,{'fixture':path},True,3)
            raw=np.load(ds.cache/'fixture.npy');im=Image.fromarray(raw)
            rnd=random.Random(int(hashlib.sha256(b'42:3:fixture').hexdigest()[:16],16))
            if rnd.random()<CFG['hflip_p']:im=ImageOps.mirror(im)
            if rnd.random()<CFG['vflip_p']:im=ImageOps.flip(im)
            im=im.rotate(rnd.uniform(-15,15),resample=Image.Resampling.BILINEAR)
            im=ImageEnhance.Brightness(im).enhance(rnd.uniform(.9,1.1));im=ImageEnhance.Contrast(im).enhance(rnd.uniform(.9,1.1))
            expected=ds.normalize(transforms.functional.to_tensor(im))
            require(torch.equal(ds[0][0],expected),'Actual brightness/contrast order sai')
        result['augmentation_determinism_bounds_no_eval_aug']='passed'
        require(choose_bn({'true':{'best_score':.80},'false':{'best_score':.805}}) is True,'BN tie rule sai')
        require(choose_bn({'true':{'best_score':.80},'false':{'best_score':.82}}) is False,'BN higher rule sai')
        require(sum(len(parameter_grid(m)) for m in METHODS)==39,'Base tuning count phải 39')
        require(39+len(METHODS)*4==79,'Base run count sai')
        for method in ('A3','A6','A8','A9','A12','SLACE'):
            g=parameter_grid(method);lo=boundary_candidate(method,g[0]);hi=boundary_candidate(method,g[-1])
            require(lo is not None and hi is not None,'Boundary missing '+method)
        result['run_budget_boundary_bn_choice']='passed'
        return result
    finally:
        CFG.clear();CFG.update(saved);OUT=old_out;RUN_SEED=old_seed


## 12. Trainer dùng cả hai GPU và AMP

Forward backbone/head chạy DataParallel trên GPU 0/1. Loss tính sau gather để không sai mẫu số CORN. Khi accumulation>1, mẫu số là toàn effective batch; batch cuối được cân theo số mẫu thực tế. FP16 chỉ cho forward model; loss tính FP32. Ghi batch nominal/mean/last thực tế, tài nguyên từng GPU và thông lượng.

In [10]:
def train_epoch(model,criterion,opt,scaler,dl,device):
    train_mode(model);num_sum=0.;den_sum=0;components=defaultdict(float);samples=0;steps=0;last_effective=0
    iterator=iter(dl);bar=tqdm(total=len(dl),desc='train',mininterval=3,leave=False,disable=not CFG.get('show_progress',True))
    while True:
        batches=[]
        for _ in range(CFG['accumulation_steps']):
            try:batches.append(next(iterator))
            except StopIteration:break
        if not batches:break
        last_effective=sum(len(y) for _,y,_ in batches)
        # CORN phải chia theo tổng conditional examples của cả effective batch, không trung bình loss từng GPU.
        denominator=sum(int((y[:,None]>=torch.arange(K-1)).sum()) if criterion.method=='CORN' else len(y) for _,y,_ in batches)
        opt.zero_grad(set_to_none=True)
        for x,y,_ in batches:
            x=x.to(device,non_blocking=True);y=y.to(device,non_blocking=True)
            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=CFG['amp'] and device.type=='cuda'):
                z=model(x)
            n,d,parts=criterion.numerator_denominator(z,y)
            require(bool(torch.isfinite(n)),'Loss NaN/Inf')
            scaler.scale(n/denominator).backward()
            num_sum+=float(n.detach());den_sum+=int(d);samples+=len(y)
            for k,v in parts.items():components[k]+=float(v.detach())
            bar.update(1)
        scaler.unscale_(opt)
        gradnorm=nn.utils.clip_grad_norm_(model.parameters(),CFG['grad_clip_norm'],error_if_nonfinite=True)
        scaler.step(opt);scaler.update();steps+=1
    bar.close()
    return {'loss':num_sum/den_sum,'components':{k:v/samples for k,v in components.items()},'samples':samples,'steps':steps,
            'effective_batch_size_mean':samples/steps,'last_effective_batch_size':last_effective}
@torch.no_grad()
def infer(model,criterion,dl,device,return_predictions=False):
    model.eval();ys=[];ps=[];ids=[];preds=[];num=0.;den=0
    for x,y,ii in dl:
        x=x.to(device,non_blocking=True);y=y.to(device,non_blocking=True)
        with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=CFG['amp'] and device.type=='cuda'):z=model(x)
        n,d,_=criterion.numerator_denominator(z,y);num+=float(n);den+=int(d)
        p=probabilities(z.float(),criterion.method)
        preds.extend((corn_decode(z.float()) if criterion.method=='CORN' else z.argmax(1)).cpu().tolist())
        ys.extend(y.cpu().tolist());ps.append(p.cpu().numpy());ids.extend(ii)
    p=np.concatenate(ps);y=np.asarray(ys)
    result=(np.asarray(ids),y,p,{'loss':num/den,**metrics(y,p,pred=preds)})
    return (*result,np.asarray(preds,dtype=np.int64)) if return_predictions else result



## 13. Vòng epoch, early stopping và resume

Chọn checkpoint bằng QWK validation trung bình trượt 3 epoch (các epoch đầu dùng số điểm hiện có), hòa chọn NLL validation thấp. Checkpoint là weights tại epoch đạt điểm đó; không trung bình weights. Scheduler theo dõi cùng điểm. Tất cả lựa chọn checkpoint, early stop và optimizer state được lưu ở ranh giới epoch.

Run completed chỉ được bỏ qua khi signature phù hợp và checkpoint/prediction cần thiết còn tồn tại. Không tự nạp weights khác cấu hình. Còn dưới thời gian dự phòng sẽ dừng chủ động với trạng thái partial. Output bị Kaggle hủy trước khi lưu thì không thể tự khôi phục; cần attach output đã được Kaggle lưu.

In [11]:
def run_training(method,fold,hp,s0,paths,weights,stage,trial=None,testing_device=None,testing_loader=None):
    if stage=='main' and fold['fold']==0:
        return reuse_fold0(method,fold,hp,s0,paths,weights,testing_device,testing_loader)
    require(stage in ('main','tuning'),'Stage không hợp lệ')
    if stage=='main':
        require(SELECTION_LOCK is not None,'Main training cần selection lock')
        validate_selection_lock(SELECTION_LOCK,s0)
        require(hp==SELECTION_LOCK['choices'][method],'Main HP khác selection lock')
    counts=np.bincount(s0['images'].loc[fold['train']].grade,minlength=K)
    key=str(trial) if stage=='tuning' else str(fold['fold'])
    container=METRIC['methods'][method]['tuning' if stage=='tuning' else 'folds']
    run_dir=OUT/('tuning' if stage=='tuning' else 'runs')/method/(f'trial_{trial}' if stage=='tuning' else f"fold_{fold['fold']}")
    run_dir.mkdir(parents=True,exist_ok=True)
    signature=digest({'protocol':PROTOCOL_HASH,'method':method,'fold':fold['fold'],'hp':hp,'stage':stage,'seed':RUN_SEED})
    record=container.setdefault(key,{'status':'not_run','hp':hp,'signature':signature,'history':[]})
    require(record['signature']==signature,'Run signature đã thay đổi; tạo output mới thay vì resume nhầm')
    if record.get('status')=='completed':
        require((run_dir/'best.pt').exists(),'Metric completed nhưng thiếu checkpoint')
        if stage!='tuning':require((run_dir/'test.npz').exists(),'Metric completed nhưng thiếu test predictions')
        compact_completed(run_dir,signature)
        LOG.info('Bỏ qua run completed: %s %s %s',method,stage,key);return record
    check_time();seed_all(RUN_SEED);device=testing_device or torch.device('cuda:0')
    require(shutil.disk_usage(OUT).free>1024**3,'Thiếu 1 GiB trống để ghi checkpoint atomically')
    make_loader=testing_loader or loader
    require(testing_device is None or str(PROTOCOL_HASH).startswith('synthetic_'),'CPU harness chỉ dành synthetic')
    model=make_model(method,weights,tiny=testing_device is not None).to(device)
    if device.type=='cuda':model=nn.DataParallel(model,device_ids=[0,1],output_device=0)
    criterion=LossContext(method,counts,hp).to(device)
    opt=torch.optim.AdamW(model.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay'])
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='max',factor=CFG['scheduler_factor'],patience=CFG['scheduler_patience'])
    scaler=scaler_new(CFG['amp'] and device.type=='cuda');start=0;best=(-float('inf'),-float('inf'));bad=0;best_epoch=-1;history=[]
    last=run_dir/'last.pt';bestpath=run_dir/'best.pt'
    if last.exists():
        require(CFG['auto_resume'],'Checkpoint tồn tại nhưng auto_resume=False')
        c=load_checkpoint(last,model,opt,scheduler,scaler,signature)
        start=c['epoch']+1;best=tuple(c['best_key']);bad=c['bad_epochs'];best_epoch=c['best_epoch'];history=c['history']
        require(bestpath.exists(),'Resume thiếu best.pt')
        LOG.info('RESUME đầy đủ %s %s epoch=%d optimizer/scheduler/scaler/RNG',method,key,start)
    record.update({'status':'training','class_weights':criterion.w.cpu().tolist(),'train_class_counts':counts.tolist(),
        'batch_size_total':CFG['batch_size'],'effective_batch_size_nominal':CFG['batch_size']*CFG['accumulation_steps'],
        'history':history,'resume_epoch':start,'freeze_bn':CFG['freeze_bn']})
    write_metric()
    try:
        for epoch in range(start,CFG['epochs']):
            if bad>=CFG['patience']:break
            check_time()
            if device.type=='cuda':
                for i in [0,1]:torch.cuda.reset_peak_memory_stats(i)
            t0=time.monotonic();lr=opt.param_groups[0]['lr']
            with Telemetry() as telemetry:
                tr=train_epoch(model,criterion,opt,scaler,make_loader(fold['train'],s0,paths,True,epoch),device)
                if device.type=='cuda':torch.cuda.synchronize()
                train_seconds=time.monotonic()-t0
                vt=time.monotonic();_,vy,vp,vm=infer(model,criterion,make_loader(fold['validation'],s0,paths),device)
                if device.type=='cuda':torch.cuda.synchronize()
                val_seconds=time.monotonic()-vt
            if device.type=='cuda':require(all(torch.cuda.max_memory_allocated(i)>0 for i in [0,1]),'Không thấy cấp phát trên cả hai GPU khi chạy model')
            require(vm['QWK'] is not None,'Validation QWK undefined')
            recent=[h['validation']['QWK'] for h in history[-(CFG['checkpoint_smoothing']-1):]] if CFG['checkpoint_smoothing']>1 else []
            score=float(np.mean(recent+[vm['QWK']]));candidate=(score,-vm['NLL'])
            improved=candidate>best
            if improved:best=candidate;best_epoch=epoch;bad=0
            else:bad+=1
            scheduler.step(score)
            row={'epoch':epoch,'lr':lr,'train':tr,'validation':vm,'checkpoint_score':score,
                'best_epoch':best_epoch,'improved':improved,'bad_epochs':bad,'train_seconds':train_seconds,
                'validation_seconds':val_seconds,'epoch_seconds':time.monotonic()-t0,
                'throughput_images_per_second':tr['samples']/train_seconds,'resources':telemetry.summary()}
            history.append(row)
            state=checkpoint_state(model,opt,scheduler,scaler,epoch,best,bad,history,signature,best_epoch)
            if improved:atomic_torch(bestpath,state)
            atomic_torch(last,state)
            record.update({'history':history,'best_epoch':best_epoch,'best_score':best[0],
                'early_stopping':bad>=CFG['patience'],'total_epoch_seconds':sum(h['epoch_seconds'] for h in history)})
            write_metric()
            LOG.info('%s | %s=%s seed=%s epoch=%d/%d lr=%.3g train=%.5f C/O=%s val=%.5f QWK=%.5f F1=%.5f Acc=%.5f GPU_GB=%s sec=%.1f %s',
                method,stage,key,RUN_SEED,epoch+1,CFG['epochs'],lr,tr['loss'],tr['components'],vm['loss'],vm['QWK'],vm['Macro_F1'],vm['Accuracy'],
                [round(torch.cuda.max_memory_allocated(i)/1024**3,2) for i in range(torch.cuda.device_count())],row['epoch_seconds'],'BEST' if improved else '')
        require(bestpath.exists(),'Không có best checkpoint')
        load_checkpoint(bestpath,model,signature=signature)
        vi,vy,vp,vm,vpred=infer(model,criterion,make_loader(fold['validation'],s0,paths),device,return_predictions=True)
        atomic_npz(run_dir/'validation.npz',ids=vi,y=vy,p=vp,pred=vpred)
        record['best_validation']=vm;record['best_score']=best[0]
        if stage!='tuning':
            require(SELECTION_LOCK['protocol_hash']==PROTOCOL_HASH and SELECTION_LOCK['complete'],'Không được test trước selection lock')
            t=fit_temperature(vy,vp)
            ti,ty,tp,tm,tpred=infer(model,criterion,make_loader(fold['test'],s0,paths),device,return_predictions=True)
            require(ti.tolist()==fold['test'],'Test ID order sai')
            pts=scaled_probs(tp,t);atomic_npz(run_dir/'test.npz',ids=ti,y=ty,p=tp,p_scaled=pts,pred=tpred,pred_scaled=decoded_predictions(pts,method))
            record.update({'temperature':t,'test':tm,'test_temperature_scaled':metrics(ty,pts,method=method)})
            if method=='CORN':
                record['test_argmax_auxiliary']=metrics(ty,tp,method=None)
        record['status']='completed';write_metric();compact_completed(run_dir,signature);return record
    except SessionLimit:
        record['status']='partial';write_metric();raise
    except Exception as e:
        record['status']='failed';record['failure_type']=type(e).__name__;write_metric();raise
    finally:
        del model,criterion,opt,scheduler,scaler;gc.collect();torch.cuda.empty_cache()



## 14. Tuning → selection lock → reuse fold 0 → folds 1–4

35 grid trials + 4 single runs + tối đa 6 boundary trials. Run A0 được tái sử dụng từ BN sanity (chỉ thêm 1 run BN ngoài ngân sách cơ bản).
Lock chứa trial, HP, signature và SHA-256 checkpoint/validation. Fold0 chỉ suy luận test; không tạo optimizer. Folds1–4 có 40 run. Tổng cơ bản 79, chưa tính sanity và mở biên.
TUNE không truy cập ảnh test; MVP không được chạy nếu chưa có lock hợp lệ.


In [12]:
def restore_previous_output(root,signature):
    if not CFG['auto_resume']:return
    candidates=[]
    for p in Path(root).rglob('metric.json'):
        try:
            m=read_json(p)
            if m.get('experiment',{}).get('protocol_hash')==signature:candidates.append((p,m))
        except (ValueError,OSError):continue
    if not candidates:return
    require(len(candidates)==1,'Nhiều output resume cùng protocol; attach chỉ phiên mới nhất')
    p,m=candidates[0]
    current=METRIC.get('experiment',{}).get('protocol_hash')
    if current:
        require(current==signature,'Output working khác protocol')
        LOG.info('Ưu tiên output working hiện tại');return
    for child in p.parent.iterdir():
        if child.name in ('cache','training.log','metric.json'):continue
        target=OUT/child.name
        require(not target.exists(),'Output local đã có dữ liệu; không ghi đè tự động')
        if child.is_dir():shutil.copytree(child,target)
        else:shutil.copy2(child,target)
    current_hardware=copy.deepcopy(METRIC['hardware']);current_tests=copy.deepcopy(METRIC.get('tests',{}))
    METRIC.clear();METRIC.update(m)
    METRIC.setdefault('hardware_sessions',[]).append(current_hardware)
    METRIC['hardware']=current_hardware;METRIC['tests']=current_tests
    write_metric();LOG.info('Đã khôi phục output phiên trước với protocol hash phù hợp')


def boundary_candidate(method,hp):
    """One extra point per tuned method, never recursively expand.

    A3 is a 2-D grid. Review does not specify which axis and beta*3 can exceed 1.
    Proposed gamma-only extension needs an explicit boundary_policy_ref before real tuning.
    """
    if method=='A3':
        require(CFG['boundary_A3_coordinate']=='gamma','A3 boundary axis chưa triển khai')
        key='gamma';grid=CFG['gamma_grid_A3']
    elif method in ('A6','A8','A9'):
        key='lambda';grid=CFG['lambda_grid_A8'] if method=='A8' else CFG['lambda_grid_A6_A9']
    elif method=='A12':key='alpha';grid=CFG['alpha_grid_A12']
    elif method=='SLACE':key='alpha';grid=CFG['slace_alpha_grid']
    else:return None
    if len(grid)<2:return None
    low,high=min(grid),max(grid);value=hp[key]
    if value not in (low,high):return None
    candidate=dict(hp);candidate[key]=value/3 if value==low else value*3
    require(candidate[key]>0,'Boundary value không hợp lệ')
    return candidate

def validate_selection_lock(lock,s0,verify_artifacts=True):
    require(lock.get('complete') is True and lock.get('protocol_hash')==PROTOCOL_HASH,'Selection lock không phù hợp')
    require(lock.get('split_sha256')==s0['summary']['split_sha256'],'Selection split hash sai')
    require(set(lock.get('choices',{}))==set(METHODS) and set(lock.get('selection_statistics',{}))==set(METHODS),'Selection lock thiếu method')
    require(lock.get('selected_on')=='fold0_validation_only','Selection source sai')
    require(lock.get('freeze_bn') is CFG['freeze_bn'],'BN selection thay đổi')
    for method in METHODS:
        stat=lock['selection_statistics'][method];trial=stat['trial']
        rec=METRIC['methods'][method]['tuning'].get(str(trial),{})
        require(rec.get('status')=='completed' and rec.get('hp')==lock['choices'][method],'Trial được chọn thiếu hoặc chưa completed')
        require(rec['signature']==stat['signature'],'Trial signature không khớp lock')
        if verify_artifacts:
            folder=OUT/'tuning'/method/f'trial_{trial}'
            require(sha_file(folder/'best.pt')==stat['checkpoint_sha256'],'Selected checkpoint thay đổi sau lock')
            require(sha_file(folder/'validation.npz')==stat['validation_sha256'],'Selected validation thay đổi sau lock')

def reuse_fold0(method,fold,hp,s0,paths,weights,testing_device=None,testing_loader=None):
    require(fold['fold']==0,'Chỉ fold0 được reuse')
    require(SELECTION_LOCK is not None,'Không được test trước selection lock')
    validate_selection_lock(SELECTION_LOCK,s0)
    stat=SELECTION_LOCK['selection_statistics'][method]
    require(hp==SELECTION_LOCK['choices'][method],'HP khác lock')
    source=OUT/'tuning'/method/f"trial_{stat['trial']}"
    dest=OUT/'runs'/method/'fold_0';dest.mkdir(parents=True,exist_ok=True)
    rec=METRIC['methods'][method]['folds'].get('0')
    if rec and rec.get('status')=='completed':
        require(rec.get('training_reused') is True,'Fold0 cũ đã train lại, không dùng')
        require(sha_file(dest/'best.pt')==stat['checkpoint_sha256'] and (dest/'test.npz').exists(),'Fold0 artifacts thiếu hoặc khác')
        return rec
    check_time();seed_all(RUN_SEED);device=testing_device or torch.device('cuda:0')
    require(testing_device is None or str(PROTOCOL_HASH).startswith('synthetic_'),'CPU harness chỉ dành synthetic')
    model=make_model(method,weights,tiny=testing_device is not None).to(device)
    if device.type=='cuda':model=nn.DataParallel(model,device_ids=[0,1])
    counts=np.bincount(s0['images'].loc[fold['train']].grade,minlength=K)
    ctx=LossContext(method,counts,hp).to(device)
    try:
        load_checkpoint(source/'best.pt',model,signature=stat['signature'])
        with np.load(source/'validation.npz',allow_pickle=False) as vv:
            require(vv['ids'].tolist()==fold['validation'],'Selected validation IDs khác fold0')
            require(np.array_equal(vv['y'],s0['images'].loc[fold['validation']].grade),'Validation labels sai')
            temperature=fit_temperature(vv['y'],vv['p'])
        # Test-only: deliberately no optimizer, train loader, or train_epoch here.
        ti,ty,tp,tm,tpred=infer(model,ctx,(testing_loader or loader)(fold['test'],s0,paths),device,return_predictions=True)
        require(ti.tolist()==fold['test'],'Fold0 test order sai')
        pts=scaled_probs(tp,temperature)
        atomic_npz(dest/'test.npz',ids=ti,y=ty,p=tp,p_scaled=pts,pred=tpred,pred_scaled=decoded_predictions(pts,method))
        shutil.copy2(source/'best.pt',dest/'best.pt')
        shutil.copy2(source/'validation.npz',dest/'validation.npz')
        selected=METRIC['methods'][method]['tuning'][str(stat['trial'])]
        rec={'status':'completed','training_reused':True,'selected_trial':stat['trial'],'hp':hp,
             'signature':stat['signature'],'checkpoint_sha256':stat['checkpoint_sha256'],
             'history':[], 'best_epoch':selected['best_epoch'],'best_score':selected['best_score'],
             'best_validation':selected['best_validation'],'temperature':temperature,'test':tm,
             'test_temperature_scaled':metrics(ty,pts,method=method),'freeze_bn':CFG['freeze_bn']}
        if method=='CORN':rec['test_argmax_auxiliary']=metrics(ty,tp)
        METRIC['methods'][method]['folds']['0']=rec;write_metric();return rec
    finally:
        del model,ctx;gc.collect();torch.cuda.empty_cache()

def execute_experiment(s0,paths,weights,testing_kwargs=None,phase='all'):
    global SELECTION_LOCK
    require(phase in ('tune','mvp','all'),'Phase sai')
    lock_path=OUT/'selection_lock.json'
    fold0=next(f for f in s0['folds'] if f['fold']==0)
    if lock_path.exists():
        SELECTION_LOCK=read_json(lock_path);validate_selection_lock(SELECTION_LOCK,s0)
    else:
        require(phase!='mvp','MVP cần selection_lock đã hoàn thành; chạy TUNE trước')
        if testing_kwargs is None:
            require(CFG['boundary_policy_ref'],'Cần xác nhận quy tắc mở biên A3 (gamma hay beta); xem README trước TUNE')
            import_bn_selected_a0(s0)
        choices={};selection_stats={};extensions=0
        for method in METHODS:
            trials=[];grid=parameter_grid(method)
            for idx,hp in enumerate(grid):
                rec=run_training(method,fold0,hp,s0,paths,weights,'tuning',idx,**(testing_kwargs or {}))
                trials.append((rec['best_score'],-rec['best_validation']['NLL'],-idx,hp))
            chosen=max(trials,key=lambda x:x[:3]);extension=None
            if CFG['boundary_enabled']:
                candidate=boundary_candidate(method,chosen[3])
                if candidate is not None:
                    require(extensions<CFG['max_boundary_trials'],'Vượt ngân sách 6 boundary trials')
                    idx=len(grid);extension={'base_winner':-chosen[2],'from':chosen[3],'candidate':candidate,'rule':'multiply_or_divide_3_once','coordinate':CFG['boundary_A3_coordinate'] if method=='A3' else ('lambda' if method in ('A6','A8','A9') else 'alpha')}
                    LOG.info('Boundary extension %s: %s',method,extension)
                    rec=run_training(method,fold0,candidate,s0,paths,weights,'tuning',idx,**(testing_kwargs or {}))
                    trials.append((rec['best_score'],-rec['best_validation']['NLL'],-idx,candidate));extensions+=1
                    chosen=max(trials,key=lambda x:x[:3])
            trial=-chosen[2];choices[method]=chosen[3]
            rec=METRIC['methods'][method]['tuning'][str(trial)];folder=OUT/'tuning'/method/f'trial_{trial}'
            selection_stats[method]={'trial':trial,'smoothed_validation_QWK':chosen[0],'validation_NLL':-chosen[1],
                'n_trials':len(trials),'boundary':extension,'signature':rec['signature'],
                'checkpoint_sha256':sha_file(folder/'best.pt'),'validation_sha256':sha_file(folder/'validation.npz')}
        SELECTION_LOCK={'complete':True,'protocol_hash':PROTOCOL_HASH,'split_sha256':s0['summary']['split_sha256'],
            'choices':choices,'selection_statistics':selection_stats,'selected_on':'fold0_validation_only',
            'freeze_bn':CFG['freeze_bn'],'boundary_trials':extensions,'boundary_policy_ref':CFG['boundary_policy_ref'],
            'locked_at':datetime.now(timezone.utc).isoformat(),'no_test_observed_during_selection':True}
        validate_selection_lock(SELECTION_LOCK,s0);atomic_json(lock_path,SELECTION_LOCK)
    METRIC['experiment']['selection_status']='locked';METRIC['experiment']['selected_hyperparameters']=SELECTION_LOCK['choices'];write_metric()
    if phase=='tune':return
    for method in METHODS:
        for fold in s0['folds']:
            run_training(method,fold,SELECTION_LOCK['choices'][method],s0,paths,weights,'main',**(testing_kwargs or {}))
    summarize_experiment(s0)


## 15. OOF, P/M, sensitivity và bootstrap

Giữ báo cáo cũ, thêm mask loại ID validation fold0 khỏi metric OOF (không sửa split). CORN dùng ordinal decoder cho raw/scaled metrics và paired bootstrap. ECE lấy xác suất của nhãn được decoder chọn; NLL dùng phân bố 5 lớp hợp lệ. Phân tích argmax chỉ là phụ.


In [13]:
def summarize_experiment(s0):
    all_predictions={};fold_rows=[]
    for m in METHODS:
        records=[]
        for f in s0['folds']:
            rec=METRIC['methods'][m]['folds'].get(str(f['fold']),{})
            if rec.get('status')!='completed':break
            path=OUT/'runs'/m/f"fold_{f['fold']}"/'test.npz'
            require(path.exists(),'Thiếu prediction đã hoàn thành; không bịa OOF')
            with np.load(path,allow_pickle=False) as x:arr={k:x[k] for k in x.files}
            records.append((f,arr))
            fold_rows.append({'method':m,'fold':f['fold'],**{k:v for k,v in rec['test'].items() if isinstance(v,(int,float)) or v is None}})
        if len(records)!=len(s0['folds']):continue
        validate_oof(s0,records)
        ids=np.concatenate([a['ids'] for _,a in records]);order=np.argsort(ids);ids=ids[order].astype(str)
        y=np.concatenate([a['y'] for _,a in records])[order];p=np.concatenate([a['p'] for _,a in records])[order]
        pt=np.concatenate([a['p_scaled'] for _,a in records])[order]
        pred=np.concatenate([a['pred'] for _,a in records])[order]
        pred_t=np.concatenate([a['pred_scaled'] for _,a in records])[order]
        all_predictions[m]=(ids,y,p,pt,pred,pred_t)
        atomic_npz(OUT/'oof'/f'{m}.npz',ids=ids,y=y,p=p,p_scaled=pt,pred=pred,pred_scaled=pred_t)
        masks=subgroup_masks(s0,ids.tolist())
        overall=metrics(y,p,pred=pred);fold_metrics=[METRIC['methods'][m]['folds'][str(f['fold'])]['test'] for f in s0['folds']]
        mean_sd={}
        for k,v in overall.items():
            vals=[r[k] for r in fold_metrics if isinstance(r.get(k),(float,int))]
            if len(vals)==len(fold_metrics):mean_sd[k]={'mean':float(np.mean(vals)),'sd':float(np.std(vals,ddof=1)) if len(vals)>1 else None}
        # OOF CI từng mô hình: bootstrap cùng dự đoán, tính trực tiếp QWK/MacroF1.
        rng=np.random.default_rng(CFG['bootstrap_seed']);boot={'QWK':[],'Macro_F1':[]}
        for _ in range(CFG['bootstrap_reps']):
            ix=rng.integers(len(y),size=len(y));r=metric_from_cm(np.bincount(y[ix]*K+pred[ix],minlength=K*K).reshape(K,K))
            for k in boot:boot[k].append(r[k])
        METRIC['oof_summary'][m]={'raw':overall,'temperature_scaled':metrics(y,pt,pred=pred_t),'mean_sd':mean_sd,
            'ci95':{k:confidence_interval(v) for k,v in boot.items()},
            'subgroups':{name:{'raw':metrics(y[mask],p[mask],pred=pred[mask]),'temperature_scaled':metrics(y[mask],pt[mask],pred=pred_t[mask])} for name,mask in masks.items()}}
        scaled_fold_metrics=[METRIC['methods'][m]['folds'][str(f['fold'])]['test_temperature_scaled'] for f in s0['folds']]
        METRIC['oof_summary'][m]['mean_sd_temperature_scaled']={}
        for key,value in METRIC['oof_summary'][m]['temperature_scaled'].items():
            values=[r[key] for r in scaled_fold_metrics if isinstance(r.get(key),(float,int))]
            if len(values)==len(scaled_fold_metrics):METRIC['oof_summary'][m]['mean_sd_temperature_scaled'][key]={'mean':float(np.mean(values)),'sd':float(np.std(values,ddof=1))}
        METRIC['methods'][m]['status']='completed';write_metric()
    if not all_predictions:return
    comparisons=[('A4','A3'),('A6','A0'),('A8','A4'),('A8','A9'),('A8','A11'),('A8','SLACE'),('A8','CORN'),('A12','A8')]
    for a,b in comparisons:
        if a not in all_predictions or b not in all_predictions:continue
        ids,y,pa,_,preda,_=all_predictions[a];bid,by,pb,_,predb,_=all_predictions[b]
        require(np.array_equal(ids,bid) and np.array_equal(y,by),'Paired OOF alignment sai')
        masks=subgroup_masks(s0,ids.tolist());res={}
        for name in ['all','M','unique_pixel_test_keep','phash4_filtered_test_keep','exclude_fold0_validation']+(['crosscheck_test_keep'] if 'crosscheck_test_keep' in masks else []):
            mask=masks[name];res[name]=paired_bootstrap(y[mask],pa[mask],pb[mask],method_a=a,method_b=b,pred_a=preda[mask],pred_b=predb[mask])
        res['cluster_all']=paired_bootstrap(y,pa,pb,s0['clusters'].loc[ids].s0_cluster_id.to_numpy(),method_a=a,method_b=b,pred_a=preda,pred_b=predb)
        METRIC['comparisons'][a+'-'+b]=res;write_metric()
    h8={'status':'blocked','reason':'requires all paired folds'}
    if 'A12-A8' in METRIC['comparisons']:
        cmp=METRIC['comparisons']['A12-A8']['all']['differences'];diffs=[];wins=0
        for f in s0['folds']:
            a=METRIC['methods']['A12']['folds'][str(f['fold'])]['test'];b=METRIC['methods']['A8']['folds'][str(f['fold'])]['test']
            diffs.append(a['Macro_F1']-b['Macro_F1']);wins+=a['QWK']>b['QWK']
        lo=cmp['QWK']['lower'];passed=len(s0['folds'])==5 and lo is not None and lo>0 and np.mean(diffs)>=0 and wins>=4
        h8={'status':'criteria_met' if passed else 'criteria_not_met','delta_qwk_ci95':cmp['QWK'],
            'mean_fold_delta_macro_f1':float(np.mean(diffs)),'winning_folds':int(wins),'required_wins':4,
            'main_method_candidate':'A12' if passed else 'A8','scope':s0['summary']['result_scope'],
            'caveat':'OOF fold-0 tuning can overlap other test folds; conditional exploratory evidence, not nested CV'}
    hypothesis_map={'H1':['A4-A3'],'H2a':['A6-A0','A8-A4'],'H2b':['A8-A9'],'H3':['A8-A4','A8-A11'],'H6':['A8-SLACE']}
    for h,pairs in hypothesis_map.items():METRIC['hypothesis_tests'][h]={'status':'evidence_available_needs_interpretation' if all(p in METRIC['comparisons'] for p in pairs) else 'partial','comparisons':pairs}
    METRIC['hypothesis_tests'].update({'H4':{'status':'not_run','reason':'DDR ngoài MVP APTOS'},
        'H5':{'status':'partial','reason':'P/M metrics and paired M computed; device probe remains outside this notebook'},
        'H7':{'status':'evidence_available_needs_interpretation','reason':'raw/scaled calibration and matched-epoch validation histories available'},
        'H8':h8,'H9':{'status':'blocked' if h8['status']!='criteria_met' else 'not_run','reason':'A13 thuộc giai đoạn 2'}})
    if all(m in all_predictions for m in METHODS):
        rank_all=sorted(METHODS,key=lambda m:METRIC['oof_summary'][m]['raw']['QWK'],reverse=True)
        rank_m=sorted(METHODS,key=lambda m:METRIC['oof_summary'][m]['subgroups']['M']['raw']['QWK'] if METRIC['oof_summary'][m]['subgroups']['M']['raw']['QWK'] is not None else -float('inf'),reverse=True)
        METRIC['hypothesis_tests']['H5'].update({'rank_all':rank_all,'rank_M':rank_m,'same_ranking':rank_all==rank_m})
    shortcut=s0['root']/'reports/shortcut_baseline_predictions.csv'
    if shortcut.exists():
        expected=read_json(s0['root']/'reproducibility/file_hashes.json')['sha256'].get('reports/shortcut_baseline_predictions.csv')
        require(sha_file(shortcut)==expected,'Shortcut prediction checksum sai')
        df=pd.read_csv(shortcut,dtype={'id_code':str});df=df[(df.scenario==s0['scenario'])&(df.partition=='test')]
        baseline={}
        for m,dfm in df.groupby('model'):
            dfm=dfm.sort_values('id_code');require(set(dfm.id_code)==s0['eligible'] and not dfm.id_code.duplicated().any(),'Metadata OOF sai')
            require(np.array_equal(dfm.grade,s0['images'].loc[dfm.id_code].grade),'Metadata labels sai')
            masks=subgroup_masks(s0,dfm.id_code.tolist());y=dfm.grade.to_numpy();pred=dfm.prediction.to_numpy()
            baseline[m]={name:metric_from_cm(np.bincount(y[mask]*K+pred[mask],minlength=K*K).reshape(K,K)) for name,mask in masks.items()}
        METRIC['metadata_baselines']=baseline
    df=pd.DataFrame(fold_rows);df.to_csv(OUT/'fold_results.csv',index=False)
    scaled_rows=[{'method':m,'fold':f['fold'],**{k:v for k,v in METRIC['methods'][m]['folds'][str(f['fold'])]['test_temperature_scaled'].items() if isinstance(v,(int,float)) or v is None}} for m in METHODS for f in s0['folds'] if METRIC['methods'][m]['folds'].get(str(f['fold']),{}).get('status')=='completed']
    pd.DataFrame(scaled_rows).to_csv(OUT/'fold_results_scaled.csv',index=False)
    rows=[]
    for m,v in METRIC['oof_summary'].items():
        row={'method':m,'OOF_QWK':v['raw']['QWK']}
        for k,s in v['mean_sd'].items():row[k+'_mean']=s['mean'];row[k+'_sd']=s['sd']
        for k,stat in v['mean_sd_temperature_scaled'].items():row['scaled_'+k+'_mean']=stat['mean'];row['scaled_'+k+'_sd']=stat['sd']
        rows.append(row)
    table=pd.DataFrame(rows).sort_values('QWK_mean',ascending=False);table.to_csv(OUT/'summary.csv',index=False)
    fig,ax=plt.subplots(figsize=(10,4));ax.bar(table.method,table.QWK_mean,yerr=table.QWK_sd,capsize=3)
    ax.set(ylabel='QWK mean ± SD across folds',title='APTOS MVP — '+s0['summary']['result_scope']);fig.tight_layout();fig.savefig(OUT/'qwk.png',dpi=160);plt.close(fig)
    write_metric()


## 16. Kiểm thử synthetic và GPU

Synthetic workflow dùng thư mục tạm, 10 tuning trials + 40 training runs + 10 fold0 inference reuses. Không phải kết quả APTOS.
GPU preflight thật ở cell tiếp theo: ResNet50 V2, 16 ảnh train fold0, mỗi loss 2 optimizer steps × 2 lần seed42; so sánh toàn bộ state_dict tuyệt đối. Kiểm tra cả hai trạng thái BN. Không hạ tiêu chí khi lỗi.


In [14]:
def synthetic_smoke(device=torch.device('cpu')):
    # Trainer dùng cùng hàm train_epoch/infer, dữ liệu giả chỉ cho kiểm thử, không đưa vào bảng kết quả APTOS.
    global PROTOCOL_HASH
    old_hash=globals().get('PROTOCOL_HASH');PROTOCOL_HASH='synthetic_test'
    saved=copy.deepcopy(CFG);results={}
    try:
        CFG.update({'amp':device.type=='cuda','batch_size':6,'accumulation_steps':2,'num_workers':0,'show_progress':False})
        for method in METHODS:
            seed_all(11);x=torch.randn(18,3,8,8);y=torch.arange(18)%K
            dataset=[(x[i],y[i],f'synthetic_{i}') for i in range(len(x))]
            dl=DataLoader(dataset,batch_size=6,shuffle=False)
            model=make_model(method,tiny=True).to(device)
            if device.type=='cuda':model=nn.DataParallel(model,device_ids=[0,1])
            ctx=LossContext(method,np.bincount(y,minlength=K),parameter_grid(method)[0]).to(device)
            opt=torch.optim.AdamW(model.parameters(),lr=.001)
            sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='max');scaler=scaler_new(CFG['amp'])
            train_epoch(model,ctx,opt,scaler,dl,device);sch.step(.1)
            _,yy,pp,r=infer(model,ctx,dl,device);require(np.isfinite(r['loss']),'Synthetic inference failed')
            with tempfile.TemporaryDirectory() as tmp:
                cp=Path(tmp)/'last.pt'
                atomic_torch(cp,checkpoint_state(model,opt,sch,scaler,0,(.1,-1),2,[],method,0))
                rand_expected=(random.random(),np.random.random(),torch.rand(2))
                train_epoch(model,ctx,opt,scaler,dl,device);sch.step(.2)
                expected={k:v.detach().cpu().clone() for k,v in bare(model).state_dict().items()}
                load_checkpoint(cp,model,opt,sch,scaler,method)
                actual=(random.random(),np.random.random(),torch.rand(2))
                require(actual[0]==rand_expected[0] and actual[1]==rand_expected[1] and torch.equal(actual[2],rand_expected[2]),'RNG resume sai')
                train_epoch(model,ctx,opt,scaler,dl,device);sch.step(.2)
                require(all(torch.equal(expected[k],v.detach().cpu()) for k,v in bare(model).state_dict().items()),'Resume khác uninterrupted')
            results[method]='passed'
    finally:
        CFG.clear();CFG.update(saved);PROTOCOL_HASH=old_hash
    return results

def dual_gpu_test():
    require(torch.cuda.is_available() and torch.cuda.device_count()>=2,'Cần hai GPU: chọn Kaggle GPU T4 ×2')
    if CFG['require_dual_t4']:require(all('T4' in torch.cuda.get_device_name(i) for i in [0,1]),'GPU hiện tại không phải T4 ×2 theo CONFIG')
    forward_seen=set();backward_seen=set();lock=threading.Lock()
    class Probe(nn.Module):
        def __init__(self):super().__init__();self.layer=nn.Linear(8,5)
        def forward(self,x):
            out=self.layer(x)
            with lock:forward_seen.add(out.device.index)
            def mark(g):
                with lock:backward_seen.add(g.device.index)
                return g
            out.register_hook(mark);return out
    model=nn.DataParallel(Probe().cuda(),device_ids=[0,1]);scaler=scaler_new(True)
    with torch.autocast('cuda',dtype=torch.float16):loss=model(torch.randn(8,8,device='cuda')).float().square().mean()
    scaler.scale(loss).backward();torch.cuda.synchronize()
    require(forward_seen==backward_seen=={0,1},'Hai GPU chưa cùng forward/backward')
    return {'status':'passed','forward_devices':sorted(forward_seen),'backward_devices':sorted(backward_seen),'amp':'passed'}

def synthetic_workflow_test():
    global OUT,METRIC,PROTOCOL_HASH,SELECTION_LOCK,START_TIME
    saved_cfg=copy.deepcopy(CFG)
    saved={k:globals().get(k) for k in ['OUT','METRIC','PROTOCOL_HASH','SELECTION_LOCK','START_TIME']}
    old_level=LOG.level
    try:
        with tempfile.TemporaryDirectory(prefix='aptos_synthetic_') as tmp:
            OUT=Path(tmp);PROTOCOL_HASH='synthetic_workflow';SELECTION_LOCK=None;START_TIME=time.monotonic()
            METRIC={'schema_version':'1.0','experiment':{'status':'synthetic_only'},'hardware':{},
                'methods':{m:{'status':'not_run','tuning':{},'folds':{}} for m in METHODS},
                'oof_summary':{},'comparisons':{},'hypothesis_tests':{},'execution_summary':{}}
            CFG.update({'epochs':1,'amp':False,'num_workers':0,'batch_size':10,'accumulation_steps':1,'show_progress':False,'bootstrap_reps':20,'boundary_enabled':False,'freeze_bn':True})
            for key in ['lambda_grid_A8','lambda_grid_A6_A9','alpha_grid_A12','beta_grid_A3','gamma_grid_A3','slace_alpha_grid']:
                CFG[key]=[CFG[key][0]]
            ids=[f'synthetic_{i:03d}' for i in range(50)];y=np.arange(50)%K
            folds=[]
            for fi in range(5):
                test=ids[fi*10:(fi+1)*10];vf=(fi+1)%5;val=ids[vf*10:(vf+1)*10]
                folds.append({'fold':fi,'train':[i for i in ids if i not in test+val],'validation':val,'test':test})
            mask_records=[{'fold':f['fold'],'id_code':i,'unique_pixel_test_keep':True,'phash4_filtered_test_keep':True} for f in folds for i in f['test']]
            s0={'images':pd.DataFrame({'grade':y},index=ids),'eligible':set(ids),'folds':folds,
                'acquisition':pd.DataFrame({'size_group':['1050x1050' if i%2 else 'synthetic_m' for i in range(50)],'acquisition_group':['synthetic_group']*50},index=ids),
                'clusters':pd.DataFrame({'s0_cluster_id':ids},index=ids),'masks':pd.DataFrame(mask_records),
                'summary':{'result_scope':'synthetic_only','split_sha256':'synthetic_only'},'root':OUT,'scenario':'S0'}
            generator=torch.Generator().manual_seed(22);xx=torch.randn(50,3,8,8,generator=generator)
            mapping={i:j for j,i in enumerate(ids)}
            def fake_loader(selected,s0,paths,training=False,epoch=0):
                if list(selected)==folds[0]['test']:
                    require(SELECTION_LOCK is not None and SELECTION_LOCK.get('complete'),'Test accessed during tuning')
                rows=[(xx[mapping[i]],int(y[mapping[i]]),i) for i in selected]
                return DataLoader(rows,batch_size=10,shuffle=training,generator=torch.Generator().manual_seed(RUN_SEED+epoch))
            LOG.setLevel(logging.WARNING)
            execute_experiment(s0,{},None,{'testing_device':torch.device('cpu'),'testing_loader':fake_loader})
            require(len(METRIC['oof_summary'])==len(METHODS),'Synthetic thiếu method OOF')
            require(all(len(v['folds'])==5 and all(r['status']=='completed' for r in v['folds'].values()) for v in METRIC['methods'].values()),'Synthetic thiếu fold')
            require(len(METRIC['comparisons'])==8,'Synthetic thiếu paired comparisons')
            require((OUT/'summary.csv').exists() and (OUT/'qwk.png').exists(),'Synthetic thiếu bảng/hình')
            # Kiểm tra bỏ qua completed + selection lock không bị thay đổi.
            lock_hash=sha_file(OUT/'selection_lock.json')
            f=folds[0];m='A0';before=sha_file(OUT/'runs'/m/'fold_0'/'best.pt')
            run_training(m,f,SELECTION_LOCK['choices'][m],s0,{},None,'main',testing_device=torch.device('cpu'),testing_loader=fake_loader)
            require(before==sha_file(OUT/'runs'/m/'fold_0'/'best.pt') and lock_hash==sha_file(OUT/'selection_lock.json'),'Completed run bị train lại')
            require(sum(r.get('training_reused',False) for v in METRIC['methods'].values() for r in v['folds'].values())==10,'Phải reuse 10 fold0')
            report={'status':'passed','methods':10,'main_training_runs':40,'reused_fold0':10,'tuning_trials':10,
                'OOF':'passed','paired_comparisons':8,'selection_lock':'passed','completed_skip':'passed',
                'metric_privacy':'passed','scope':'synthetic temporary data only; deleted after test'}
            return report
    finally:
        CFG.clear();CFG.update(saved_cfg);globals().update(saved);LOG.setLevel(old_level)


## 17. Đăng ký phiên bản, GPU preflight, sanity và chạy workflow

Cell định nghĩa các hàm hỗ trợ rồi gọi `main()` theo CONFIG. CPU_TESTS không cần dữ liệu/Git/GPU.
SANITY yêu cầu registration đã xác minh; chỉ chạy `sanity_tasks` đã chọn. TUNE/MVP yêu cầu BN lock và GPU evidence cùng phiên bản/môi trường.
Nếu native CE/NLL báo lỗi deterministic CUDA, lưu lỗi, đổi `a0_ce_backend='logsoftmax_gather'`, chạy lại CPU/GPU preflight và đăng ký cấu hình mới. Không tự fallback trong run.


In [15]:
def code_object_signature(code):
    def constant(v):
        if isinstance(v,types.CodeType):return code_object_signature(v)
        if isinstance(v,tuple):return [constant(x) for x in v]
        if isinstance(v,frozenset):return sorted(repr(x) for x in v)
        return repr(v)
    return {'bytecode':code.co_code.hex(),'constants':[constant(v) for v in code.co_consts],
            'names':code.co_names,'varnames':code.co_varnames,'freevars':code.co_freevars,
            'cellvars':code.co_cellvars,'args':code.co_argcount,'kwonly':code.co_kwonlyargcount,
            'posonly':code.co_posonlyargcount,'flags':code.co_flags,'exceptions':getattr(code,'co_exceptiontable',b'').hex()}

def verify_live_code(source):
    """Compile committed export on this Python version, compare every live function/method.
    Does not execute source, does not trust an editable hardcoded commit string.
    """
    module=compile(source,'<registered-code>','exec');checked=0
    tree=ast.parse(source)
    nodes={x.name:x for x in tree.body if isinstance(x,(ast.FunctionDef,ast.ClassDef))}
    constants={c.co_name:c for c in module.co_consts if isinstance(c,types.CodeType)}
    for name,node in nodes.items():
        actual=globals().get(name);expected=constants[name]
        if isinstance(node,ast.FunctionDef):
            require(callable(actual) and hasattr(actual,'__code__'),'Missing live function '+name)
            # torch.no_grad decorates infer; compare wrapped function body.
            original=inspect.unwrap(actual)
            require(code_object_signature(original.__code__)==code_object_signature(expected),'Running code differs from commit: '+name)
            checked+=1
        else:
            require(isinstance(actual,type),'Missing live class '+name)
            methods={c.co_name:c for c in expected.co_consts if isinstance(c,types.CodeType)}
            for child in node.body:
                if isinstance(child,ast.FunctionDef):
                    method=inspect.unwrap(getattr(actual,child.name))
                    require(code_object_signature(method.__code__)==code_object_signature(methods[child.name]),'Running method differs from commit: '+name+'.'+child.name)
                    checked+=1
    return checked

def verify_registration():
    import re
    path=CFG.get('registration_path') or os.environ.get('APTOS_REGISTRATION')
    require(path and Path(path).is_file(),'Thiếu runtime_registration.json; commit code rồi chạy src/register_version.py')
    reg=read_json(path)
    require(re.fullmatch(r'[0-9a-f]{40}',reg.get('git_commit','')) is not None,'Mã commit phải là SHA Git thật 40 ký tự')
    require(hashlib.sha256(reg['code_source'].encode()).hexdigest()==reg['code_file_sha256'],'Registration source checksum sai')
    require(reg.get('code_sha256')==NOTEBOOK_CODE_SHA256,'Registration thuộc bản code khác')
    checked=verify_live_code(reg['code_source'])
    require(reg.get('remote_verified') is True,'Commit chưa được xác nhận trên remote GitHub')
    require(reg.get('pytest_log_sha256'),'Thiếu log pytest đã commit')
    require(reg.get('methods')==list(METHODS) and reg.get('K')==K,'Methods/K khác commit')
    if CFG['git_commit'] is not None:require(CFG['git_commit']==reg['git_commit'],'CFG commit khác registration')
    CFG['git_commit']=reg['git_commit']
    return {k:reg[k] for k in ['git_commit','code_file_sha256','notebook_file_sha256','pytest_log_sha256','remote_verified']}|{'live_functions_verified':checked}

def protocol_config():
    excluded={'execution_mode','input_root','output_root','auto_resume','session_hours','stop_margin_minutes',
              'split_sha256_selector','registration_path','sanity_tasks','kaggle_version','git_commit','boundary_policy_ref'}
    return {k:v for k,v in CFG.items() if k not in excluded}

def base_evidence_hash(s0,weights):
    config=protocol_config();config.pop('freeze_bn',None)
    return digest({'config':config,'code':NOTEBOOK_CODE_SHA256,'split':s0['summary']['split_sha256'],
                   'weights':sha_file(weights) if weights else None,'seed':RUN_SEED})

def official_config_check():
    require(CFG['allow_exploratory_unapproved'] is False and CFG['allow_provisional_protocol'] is False,'Official flags phải False')
    require(CFG['protocol_approved_ref']=='Review-2026-11-07 – QĐ24, QĐ25','Thiếu phê duyệt bắt buộc')
    expected={'backbone':'resnet50','pretrained':True,'input_size':384,'batch_size':32,'accumulation_steps':1,
        'epochs':50,'patience':10,'lr':1e-4,'weight_decay':1e-4,'optimizer':'AdamW','scheduler':'ReduceLROnPlateau',
        'scheduler_factor':.5,'scheduler_patience':3,'amp':True,'grad_clip_norm':5.,'seed':42,
        'checkpoint_smoothing':3,'sqrt_beta':.999,'sqrt_gamma':2.,'brightness':.1,'contrast':.1,'freeze_bn_affine':False}
    for k,v in expected.items():require(CFG[k]==v,'Config khác QĐ25: '+k)
    require(tuple(map(int,version('scikit-learn').split('.')[:2]))>=(1,8),'Cần scikit-learn >=1.8')

def save_run_snapshot(s0,weights,registration):
    snapshot={'config':CFG,'protocol_hash':PROTOCOL_HASH,'code_sha256':NOTEBOOK_CODE_SHA256,
        'git':registration,'split_sha256':s0['summary']['split_sha256'],
        's0_manifest_sha256':sha_file(s0['root']/'reproducibility/s0_version_manifest.json'),
        'weights_sha256':sha_file(weights) if weights else None,'hardware':hardware_info(),
        'time_utc':datetime.now(timezone.utc).isoformat()}
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    atomic_json(OUT/'provenance'/f'{stamp}_{PROTOCOL_HASH}.json',snapshot)

def preflight_two_steps(method,s0,paths,weights,freeze_bn,seed=42):
    require(torch.cuda.is_available(),'GPU preflight cần CUDA thật')
    fold=next(f for f in s0['folds'] if f['fold']==0);ids=fold['train'][:16]
    require(len(ids)==16,'Cần 16 ảnh train thật')
    saved=CFG['freeze_bn'];CFG['freeze_bn']=freeze_bn
    try:
        seed_all(seed);model=make_model(method,weights).to('cuda:0')
        model=nn.DataParallel(model,device_ids=[0,1])
        ctx=LossContext(method,np.bincount(s0['images'].loc[fold['train']].grade,minlength=K),parameter_grid(method)[0]).to('cuda:0')
        opt=torch.optim.AdamW(model.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay']);scaler=scaler_new(True)
        # same real 16-image batch, two optimizer steps; no validation/test access.
        dl=loader(ids,s0,paths,True,0)
        require(len(dl)==1,'Preflight phải 1 batch 16 ảnh')
        for _ in range(2):
            result=train_epoch(model,ctx,opt,scaler,dl,torch.device('cuda:0'))
            require(result['steps']==1 and result['samples']==16,'Preflight sai optimizer step count')
        torch.cuda.synchronize()
        require(len(opt.state)==sum(p.requires_grad for p in model.parameters()),'AMP bỏ qua mọi optimizer step')
        steps=[int(v['step'].item()) for v in opt.state.values()]
        require(all(v==2 for v in steps),'AMP đã bỏ qua step: preflight chưa đạt 2 optimizer steps')
        return {k:v.detach().cpu().clone() for k,v in bare(model).state_dict().items()}
    finally:
        CFG['freeze_bn']=saved
        if 'model' in locals():del model,ctx,opt,scaler
        gc.collect();torch.cuda.empty_cache()

def gpu_preflight(s0,paths,weights):
    evidence=base_evidence_hash(s0,weights)
    report={'status':'running','evidence_hash':evidence,'hardware':hardware_info(),'seed':42,'samples':16,'optimizer_steps':2,'methods':{}}
    target=OUT/'checks'/'gpu_preflight.json';atomic_json(target,report)
    try:
        dual_gpu_test()
        for frozen in (True,False):
            for method in METHODS:
                key=method+('_bn_frozen' if frozen else '_bn_train')
                a=preflight_two_steps(method,s0,paths,weights,frozen)
                b=preflight_two_steps(method,s0,paths,weights,frozen)
                different=[k for k in a if not torch.equal(a[k],b[k])]
                require(not different,'GPU deterministic mismatch '+key+': '+str(different[:5]))
                report['methods'][key]={'status':'passed','exact_state_equal':True,'state_tensors':len(a)}
                del a,b;atomic_json(target,report)
        report['status']='passed'
    except Exception as exc:
        report['status']='failed';report['error_type']=type(exc).__name__;report['error']=str(exc)
        raise
    finally:atomic_json(target,report)
    return report

def require_gpu_evidence(s0,weights):
    file=OUT/'checks'/'gpu_preflight.json'
    require(file.is_file(),'Chạy GPU_PREFLIGHT trước')
    r=read_json(file);require(r.get('status')=='passed' and r.get('evidence_hash')==base_evidence_hash(s0,weights),'GPU preflight thiếu/failed/khác config hoặc source')
    old=r['hardware'];now=hardware_info()
    for key in ('torch','torchvision','cuda','driver'):
        require(old[key]==now[key],'GPU preflight khác môi trường '+key)
    require(old['gpus']==now['gpus'],'GPU preflight khác GPU')

# Isolate sanity experiments without changing the main protocol or result containers.
from contextlib import contextmanager
@contextmanager
def isolated_run(folder,signature,updates):
    global OUT,METRIC,PROTOCOL_HASH,START_TIME
    saved={k:globals().get(k) for k in ('OUT','METRIC','PROTOCOL_HASH','START_TIME')};config=copy.deepcopy(CFG)
    try:
        OUT=Path(folder);OUT.mkdir(parents=True,exist_ok=True);PROTOCOL_HASH=signature;START_TIME=time.monotonic()
        CFG.update(updates)
        file=OUT/'metric.json'
        METRIC=read_json(file) if file.exists() else {'schema_version':'1.0','experiment':{'status':'sanity'},'hardware':{},
            'methods':{m:{'status':'not_run','tuning':{},'folds':{}} for m in METHODS},'oof_summary':{},'comparisons':{},'hypothesis_tests':{},'execution_summary':{}}
        yield
    finally:
        CFG.clear();CFG.update(config);globals().update(saved)

def choose_bn(records):
    a,b=records['true']['best_score'],records['false']['best_score']
    require(np.isfinite(a) and np.isfinite(b),'BN scores không hữu hạn')
    return True if abs(a-b)<.01 else a>b

def sanity_bn(s0,paths,weights):
    root=OUT;fold=next(f for f in s0['folds'] if f['fold']==0);base=base_evidence_hash(s0,weights)
    records={};artifacts={}
    for frozen in (True,False):
        key=str(frozen).lower();folder=root/'sanity'/'bn'/key;signature=digest({'base':base,'freeze_bn':frozen})
        with isolated_run(folder,signature,{'freeze_bn':frozen}):
            rec=run_training('A0',fold,{},s0,paths,weights,'tuning',0)
            records[key]=copy.deepcopy(rec)
            source=OUT/'tuning'/'A0'/'trial_0'
            artifacts[key]={'protocol_hash':signature,'signature':rec['signature'],
                'checkpoint_sha256':sha_file(source/'best.pt'),'validation_sha256':sha_file(source/'validation.npz')}
    chosen=choose_bn(records)
    lock={'status':'passed','evidence_hash':base,'freeze_bn':chosen,'rule':'higher best smoothed QWK; abs difference <0.01 => True',
          'records':records,'artifacts':artifacts,'selected_at':datetime.now(timezone.utc).isoformat()}
    atomic_json(root/'bn_selection.json',lock)
    return {'status':'passed','freeze_bn':chosen,'scores':{k:r['best_score'] for k,r in records.items()}}

def apply_bn_lock(s0,weights):
    p=OUT/'bn_selection.json';require(p.is_file(),'Chạy SANITY bn trước tuning')
    lock=read_json(p)
    require(lock.get('status')=='passed' and lock['evidence_hash']==base_evidence_hash(s0,weights),'BN lock khác dữ liệu/code/config')
    require(lock['freeze_bn']==choose_bn(lock['records']),'BN decision sai quy tắc')
    require(CFG['freeze_bn'] is None or CFG['freeze_bn'] is lock['freeze_bn'],'CFG freeze_bn khác lựa chọn đã khóa')
    CFG['freeze_bn']=lock['freeze_bn']
    return lock

def import_bn_selected_a0(s0):
    lock=read_json(OUT/'bn_selection.json');key=str(CFG['freeze_bn']).lower()
    require(lock['freeze_bn'] is CFG['freeze_bn'],'BN choice khác')
    source=OUT/'sanity'/'bn'/key/'tuning'/'A0'/'trial_0';e=lock['artifacts'][key]
    require(sha_file(source/'best.pt')==e['checkpoint_sha256'] and sha_file(source/'validation.npz')==e['validation_sha256'],'BN artifacts đã thay đổi')
    dest=OUT/'tuning'/'A0'/'trial_0';dest.mkdir(parents=True,exist_ok=True)
    signature=digest({'protocol':PROTOCOL_HASH,'method':'A0','fold':0,'hp':{},'stage':'tuning','seed':RUN_SEED})
    if '0' in METRIC['methods']['A0']['tuning']:
        require(METRIC['methods']['A0']['tuning']['0']['signature']==signature,'A0 trial khác protocol')
        return
    checkpoint=torch.load(source/'best.pt',map_location='cpu',weights_only=False)
    require(checkpoint['protocol_hash']==e['protocol_hash'] and checkpoint['signature']==e['signature'],'BN checkpoint signature sai')
    # Only provenance metadata changes. Model tensors remain identical; no training.
    checkpoint.update({'protocol_hash':PROTOCOL_HASH,'signature':signature,'reused_bn_source_sha256':e['checkpoint_sha256']})
    atomic_torch(dest/'best.pt',checkpoint);shutil.copy2(source/'validation.npz',dest/'validation.npz')
    record=copy.deepcopy(lock['records'][key]);record.update({'signature':signature,'reused_bn_sanity':True})
    METRIC['methods']['A0']['tuning']['0']=record;write_metric()

def sanity_overfit(s0,paths,weights):
    root=OUT;fold=next(f for f in s0['folds'] if f['fold']==0);ids=fold['train'][:32]
    require(len(ids)==32,'Overfit cần 32 ảnh train');results={}
    saved=CFG['freeze_bn'];CFG['freeze_bn']=True if saved is None else saved
    try:
        for method in ('A0','A8'):
            seed_all(RUN_SEED);model=nn.DataParallel(make_model(method,weights).cuda(),device_ids=[0,1])
            hp=parameter_grid(method)[0] if method=='A0' else {'beta':CFG['sqrt_beta'],'gamma':CFG['sqrt_gamma'],'lambda':1.0}
            ctx=LossContext(method,np.bincount(s0['images'].loc[fold['train']].grade,minlength=K),hp).cuda()
            opt=torch.optim.AdamW(model.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay']);scaler=scaler_new(True)
            dl=loader(ids,s0,paths,False);history=[]
            for step in range(200):
                check_time();r=train_epoch(model,ctx,opt,scaler,dl,torch.device('cuda:0'))
                history.append({'step':step+1,'loss':r['loss']})
            _,_,_,result=infer(model,ctx,dl,torch.device('cuda:0'))
            # Operational near-zero tolerance is explicitly logged, not a research selection rule.
            optimizer_steps=min(int(v['step'].item()) for v in opt.state.values()) if opt.state else 0
            results[method]={'status':'passed' if result['Accuracy']==1.0 and result['loss']<=CFG['sanity_overfit_loss_tolerance'] and optimizer_steps==200 else 'failed',
                'loss':result['loss'],'accuracy':result['Accuracy'],'hp':hp,'augmentation':False,'history':history,'optimizer_steps':optimizer_steps,'loss_tolerance':CFG['sanity_overfit_loss_tolerance'],'freeze_bn':CFG['freeze_bn']}
            atomic_json(root/'checks'/'overfit.json',{'evidence_hash':base_evidence_hash(s0,weights),'methods':results})
            del model,ctx,opt,scaler;gc.collect();torch.cuda.empty_cache()
        return results
    finally:CFG['freeze_bn']=saved

def sanity_shuffled(s0,paths,weights):
    fold=next(f for f in s0['folds'] if f['fold']==0);shuffled=copy.copy(s0);shuffled['images']=s0['images'].copy()
    ids=fold['train'];labels=shuffled['images'].loc[ids,'grade'].to_numpy(copy=True)
    shuffled['images'].loc[ids,'grade']=np.random.default_rng(RUN_SEED).permutation(labels)
    root=OUT;signature=digest({'base':base_evidence_hash(s0,weights),'sanity':'shuffled'})
    with isolated_run(root/'sanity'/'shuffled',signature,{'epochs':10,'patience':11,'freeze_bn':True if CFG['freeze_bn'] is None else CFG['freeze_bn']}):
        rec=run_training('A0',fold,{},shuffled,paths,weights,'tuning',0)
        require(len(rec['history'])==10,'Shuffled sanity chưa đủ 10 epoch')
        qwk=rec['history'][-1]['validation']['QWK']
        result={'status':'passed' if -.1<=qwk<=.1 else 'failed','epoch10_QWK':qwk,
            'best_checkpoint_QWK':rec['best_validation']['QWK'],'all_epochs_QWK':[h['validation']['QWK'] for h in rec['history']]}
    result['evidence_hash']=base_evidence_hash(s0,weights)
    atomic_json(root/'checks'/'shuffled.json',result);return result

def runtime_benchmark():
    p=OUT/'bn_selection.json';require(p.is_file(),'Cần BN runs thật trước ước lượng runtime')
    r=read_json(p);estimates={}
    for key,rec in r['records'].items():
        seconds=[h['epoch_seconds'] for h in rec['history']]
        require(seconds,'Không có thời gian thực đo')
        wall=sum(seconds)/3600
        estimates[key]={'measured_epochs':len(seconds),'early_stopping_reached':rec['early_stopping'],
            'epochs_to_early_stopping':len(seconds) if rec['early_stopping'] else None,
            'measured_epoch_seconds':seconds,'run_wall_hours':wall,
            'projected_79_run_wall_hours':79*wall,'projected_79_run_device_hours':79*wall*torch.cuda.device_count(),
            'projection_assumption':'79 runs have same duration as measured A0 run; excludes preflight/sanity/boundary trials'}
    atomic_json(OUT/'checks'/'runtime.json',estimates);return estimates

def acquisition_probe(s0,paths,weights):
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.dummy import DummyClassifier
    from sklearn.exceptions import ConvergenceWarning
    seed_all(RUN_SEED);ids=sorted(s0['eligible']);device=torch.device('cuda:0')
    model=make_model('A0',weights);model.fc=nn.Identity();model.requires_grad_(False);model.eval();model=model.to(device)
    features=[];seen=[]
    try:
        with torch.no_grad():
            for x,_,ii in loader(ids,s0,paths,False):
                features.append(model(x.to(device)).float().cpu().numpy());seen.extend(ii)
        require(seen==ids,'Probe feature IDs sai')
        xx=np.concatenate(features);index={iid:j for j,iid in enumerate(ids)};target=s0['acquisition'].loc[ids].size_group.astype(str).to_numpy()
        atomic_npz(OUT/'sanity'/'probe_features.npz',ids=np.asarray(ids),features=xx,size_group=target)
        rows=[]
        for fold in s0['folds']:
            tr=np.array([index[i] for i in fold['train']]);te=np.array([index[i] for i in fold['test']])
            linear=make_pipeline(StandardScaler(),LogisticRegression(C=1.,max_iter=3000,random_state=RUN_SEED))
            majority=DummyClassifier(strategy='most_frequent')
            with warnings.catch_warnings():
                warnings.simplefilter('error',ConvergenceWarning);linear.fit(xx[tr],target[tr])
            majority.fit(xx[tr],target[tr])
            pred=linear.predict(xx[te]);base=majority.predict(xx[te])
            rows.append({'fold':fold['fold'],'accuracy':float((pred==target[te]).mean()),
                'majority_accuracy':float((base==target[te]).mean()),'n_test':len(te)})
        # Read preregistered train-only metadata baselines and recompute on P/M.
        file=s0['root']/'reports/shortcut_baseline_predictions.csv'
        require(file.is_file(),'Thiếu shortcut_baseline_predictions.csv để báo cáo mức sàn P/M')
        hashes=read_json(s0['root']/'reproducibility/file_hashes.json')['sha256']
        require(sha_file(file)==hashes.get('reports/shortcut_baseline_predictions.csv'),'Metadata baseline hash sai')
        df=pd.read_csv(file,dtype={'id_code':str});df=df[(df.scenario==s0['scenario'])&(df.partition=='test')]
        baselines={}
        for name,part in df.groupby('model'):
            require(not part.id_code.duplicated().any() and set(part.id_code)==s0['eligible'],'Metadata coverage sai')
            require(np.array_equal(part.grade,s0['images'].loc[part.id_code].grade),'Metadata labels sai')
            masks=subgroup_masks(s0,part.id_code.tolist());y=part.grade.to_numpy();p=part.prediction.to_numpy()
            baselines[name]={k:metric_from_cm(np.bincount(y[v]*K+p[v],minlength=K*K).reshape(K,K)) for k,v in masks.items() if k in ('all','P','M')}
        result={'status':'completed','feature_backbone':'frozen ResNet50 ImageNet-1K V2; crop/pad/resize/normalize; no augmentation',
            'probe':'fixed C=1 standardized logistic regression; train-only fit; S0 tests; no hyperparameter tuning',
            'folds':rows,'metadata_baselines':baselines}
        atomic_json(OUT/'checks'/'acquisition_probe.json',result);return result
    finally:del model;gc.collect();torch.cuda.empty_cache()

def run_sanity(s0,paths,weights):
    tasks=CFG['sanity_tasks'];require(tasks,'Chọn sanity_tasks; không tự chạy tất cả sanity checks')
    functions={'overfit':sanity_overfit,'shuffled':sanity_shuffled,'bn':sanity_bn,'probe':acquisition_probe}
    results={}
    for task in tasks:
        require(task in (*functions,'runtime'),'Sanity task không hợp lệ')
        results[task]=runtime_benchmark() if task=='runtime' else functions[task](s0,paths,weights)
    return results

def require_sanity_evidence(s0,weights):
    base=base_evidence_hash(s0,weights)
    for name in ('overfit','shuffled'):
        file=OUT/'checks'/(name+'.json');require(file.is_file(),'Thiếu sanity check '+name)
        report=read_json(file);require(report.get('evidence_hash')==base,'Sanity evidence khác config/source: '+name)
        if name=='overfit':
            require(set(report.get('methods',{}))=={'A0','A8'} and all(r.get('status')=='passed' for r in report['methods'].values()),'Overfit one batch chưa đạt')
        else:require(report.get('status')=='passed','Shuffled labels chưa đạt')

def restore_evidence_output(s0,weights):
    if not CFG['auto_resume']:return
    signature=base_evidence_hash(s0,weights);candidates=[]
    for p in Path(CFG['input_root']).rglob('gpu_preflight.json'):
        try:
            obj=read_json(p)
            if obj.get('evidence_hash')==signature and p.parent.name=='checks':candidates.append(p.parent.parent)
        except (OSError,ValueError):continue
    if not candidates:return
    require(len(set(candidates))==1,'Attach chỉ một output trước cho cùng evidence hash')
    previous=candidates[0]
    if previous.resolve()==OUT.resolve():return
    # Existing local evidence is authoritative for a restarted kernel in the same session.
    if (OUT/'checks'/'gpu_preflight.json').exists():return
    for child in previous.iterdir():
        if child.name in ('cache','training.log','metric.json'):continue
        target=OUT/child.name;require(not target.exists(),'Resume conflict: '+child.name)
        if child.is_dir():shutil.copytree(child,target)
        else:shutil.copy2(child,target)
    if (previous/'metric.json').is_file():
        old=read_json(previous/'metric.json');tests=METRIC.get('tests',{});hardware=METRIC['hardware']
        METRIC.clear();METRIC.update(old);METRIC['tests']=tests;METRIC['hardware']=hardware
        write_metric()

def main():
    global RUN_SEED,PROTOCOL_HASH,KNOWN_IDS
    mode=CFG['execution_mode']
    require(mode in ('CPU_TESTS','GPU_PREFLIGHT','SANITY','TUNE','MVP','RUN_ALL'),'execution_mode không hợp lệ')
    initialize_output()
    try:
        METRIC['hardware']=hardware_info();RUN_SEED=CFG['seed'];seed_all(RUN_SEED)
        METRIC['tests']['losses']=run_unit_tests()
        METRIC['tests']['fixes']=run_fix_tests()
        METRIC['tests']['synthetic_cpu_resume']=synthetic_smoke()
        METRIC['tests']['synthetic_full_workflow']=synthetic_workflow_test()
        write_metric()
        if mode=='CPU_TESTS':
            METRIC['experiment']['status']='cpu_tests_passed_gpu_not_run';return
        require(CFG['batch_size']==32 and CFG['accumulation_steps']==1,'QĐ25 yêu cầu effective batch 32')
        require(CFG['slace_variant']=='prox_max','SLACE variant không hợp lệ')
        root,split,manifest=discover_s0(CFG['input_root']);s0=audit_s0(root,split,manifest)
        KNOWN_IDS=set(s0['images'].index)
        source,paths=find_source(CFG['input_root'],s0);s0=audit_s0(root,split,manifest,source)
        weights=resolve_weights(CFG['input_root']);restore_evidence_output(s0,weights)
        METRIC['experiment'].update({'data':s0['summary'],'seed':RUN_SEED,'code_sha256':NOTEBOOK_CODE_SHA256,
            'reference_sha256':REFERENCE_SHA256,'result_scope':'preflight_or_sanity_only'})
        if mode=='GPU_PREFLIGHT':
            METRIC['tests']['gpu_preflight']=gpu_preflight(s0,paths,weights)
            METRIC['experiment']['status']='gpu_preflight_passed';return
        official_config_check();registration=verify_registration()
        require_gpu_evidence(s0,weights)
        if mode=='SANITY':
            # Sanity evidence is separate and never enters main model results.
            PROTOCOL_HASH=digest({'base':base_evidence_hash(s0,weights),'stage':'sanity'})
            save_run_snapshot(s0,weights,registration)
            METRIC['tests']['sanity']=run_sanity(s0,paths,weights)
            METRIC['experiment']['status']='sanity_completed_review_results';return
        require(s0['summary']['result_scope']=='approved','MVP cần manifest approved_frozen')
        require(len(next(f for f in s0['folds'] if f['fold']==0)['validation'])==358,'Fold0 validation khác S0-v1.0 đã duyệt')
        require(s0['summary']['split_sha256']==CFG['approved_split_sha256'],'Không đúng hash S0 đã duyệt')
        require_sanity_evidence(s0,weights)
        apply_bn_lock(s0,weights)
        protocol=protocol_config()
        PROTOCOL_HASH=digest({'config':protocol,'code':NOTEBOOK_CODE_SHA256,'seed':RUN_SEED,
            'split':manifest['splits_json_sha256'],'weights':sha_file(weights) if weights else None})
        previous=METRIC['experiment'].get('protocol_hash')
        require(previous in (None,PROTOCOL_HASH),'Output đã có protocol khác; dùng output_root mới')
        save_run_snapshot(s0,weights,registration)
        METRIC['experiment'].update({'status':'running','protocol_hash':PROTOCOL_HASH,'training_config':protocol,
            'git_commit':CFG['git_commit'],'kaggle_version':CFG['kaggle_version'],
            'pretrained_weights_sha256':sha_file(weights) if weights else None,'data':s0['summary'],'result_scope':'approved',
            'reference_implementation_status':'embedded_source_value_and_gradient_verified_with_documented_CNN_adaptation'})
        write_metric();execute_experiment(s0,paths,weights,phase={'TUNE':'tune','MVP':'mvp','RUN_ALL':'all'}[mode])
        METRIC['experiment']['status']='tuning_locked' if mode=='TUNE' else 'completed'
    except SessionLimit as e:
        METRIC['experiment']['status']='partial';LOG.warning('%s',e)
    except Exception as e:
        METRIC['experiment']['status']='blocked' if isinstance(e,Blocked) else 'failed'
        METRIC['execution_summary']['failure_type']=type(e).__name__;LOG.exception('Dừng: %s',e);raise
    finally:
        runs=[r for m in METRIC['methods'].values() for stage in ('tuning','folds') for r in m[stage].values()]
        METRIC['execution_summary'].update({'completed_records':sum(r.get('status')=='completed' for r in runs),
            'completed_training_runs':sum(r.get('status')=='completed' and not r.get('training_reused',False) for r in runs),
            'reused_fold0_records':sum(r.get('training_reused',False) for r in runs),
            'failed_runs':sum(r.get('status')=='failed' for r in runs),
            'total_runtime_seconds':METRIC['execution_summary'].get('total_runtime_seconds',0)+time.monotonic()-START_TIME})
        update_resource_totals(runs);write_metric()
        LOG.info('Trạng thái: %s',METRIC['experiment']['status'])

if __name__=='__main__':
    main()


SHA256 ảnh nguồn:   0%|          | 0/3662 [00:00<?, ?it/s]

2026-11-10 14:08:30,957 | INFO | Đang tự tải ResNet50 ImageNet-1K V2 từ PyTorch: https://download.pytorch.org/models/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 213MB/s]


2026-11-10 14:08:32,104 | INFO | Đã tải và kiểm SHA-256 ResNet50 ImageNet-1K V2: /kaggle/working/APTOS_MVP_S0v1/cache/pretrained/resnet50-11ad3fa6.pth


train:   0%|          | 0/1 [00:00<?, ?it/s]

train:   0%|          | 0/1 [00:00<?, ?it/s]

... (75 outputs left out)

train:   0%|          | 0/1 [00:00<?, ?it/s]

train:   0%|          | 0/1 [00:00<?, ?it/s]

train:   0%|          | 0/1 [00:00<?, ?it/s]

2026-11-10 14:10:12,785 | INFO | Trạng thái: gpu_preflight_passed


## 18. Các sản phẩm sau khi chạy

- `metric.json`: một tệp metric tổng hợp, cập nhật liên tục.
- `training.log`, `selection_lock.json`.
- `tuning/` và `runs/`: best checkpoint; last checkpoint đầy đủ chỉ còn ở run đang dở; validation/test NPZ riêng.
- `oof/`: NPZ riêng cho từng method; chỉ tạo khi đủ fold và kiểm tra coverage/alignment đạt.
- `fold_results.csv`, `summary.csv`, `qwk.png`.
- `cache/`: ảnh đã crop/resize để giảm CPU; có thể bỏ khi chuyển phiên vì tự tạo lại được.

Tổng kích thước `metric.json` có giới hạn cấu hình; nếu vượt thì báo lỗi, không tự xóa epoch history. Các tệp checkpoint/OOF có dữ liệu khác với metric và không bị nhúng vào metric.

## 19. Kiểm chứng bàn giao

Xem `logs/pytest.log`, `logs/environment.json` và `FIX_REPORT.md` trong gói hỗ trợ để biết chính xác kiểm thử đã chạy. Không kế thừa tuyên bố Passed của notebook cũ.
Không có kết quả huấn luyện APTOS, checkpoint thật hay kết quả GPU được tạo sẵn trong notebook.


## 20. Kaggle và resume

1. Chạy CPU_TESTS; attach APTOS gốc và S0 nguyên cấu trúc khi chuyển sang GPU.
2. Chọn T4×2, Internet On nếu cần tải pretrained V2. Chạy GPU_PREFLIGHT; kiểm tra `checks/gpu_preflight.json`.
3. Commit code + tests + log; chạy `src/register_version.py` để tạo registration bên ngoài code sau commit, attach JSON vào Kaggle và đặt `registration_path`. Không sửa mã commit vào notebook rồi commit lại.
4. Chọn SANITY, chạy từng tác vụ. BN comparison chạy đầy đủ A0 True/False và giữ checkpoint được chọn làm trial0 của A0.
5. Xác nhận quy tắc biên A3; cập nhật metadata phê duyệt S0 đúng QĐ24. Chạy TUNE rồi MVP.
6. Resume: attach duy nhất output mới nhất gồm checks, bn_selection, sanity, tuning, runs, selection_lock, metric; giữ nguyên config. Code không sửa S0.
